<div class="blog-language-switch" role="group" aria-label="Article language">
<span aria-current="page">English</span>
<a href="/ipynb/zh-CN/NLP/05-training-optimization.html" lang="zh-CN" hreflang="zh-CN">中文</a>
</div>

[Back to NLP guideline](Natural-Language-Processing.html)


## **Training, Optimization, and Post-Training** {#training-optimization}

Training turns a model architecture into a learned system. The architecture determines which computations are possible, while the data, objective, optimizer, and update schedule determine which behavior is actually acquired. A model may have enough capacity to solve a task and still fail because the target is poorly specified, the gradients are unstable, or the examples do not represent deployment conditions.

For a small classifier, training may mean learning a boundary between positive and negative reviews. For a language model, it may begin with next-token prediction over a large corpus, continue with supervised instruction examples, and end with preference optimization that changes which plausible response the model tends to choose. These stages use related mathematics, but they optimize different signals.

The core gradient-based loop remains:

```text
mini-batch
-> forward pass
-> scalar loss
-> backward pass
-> optimizer update
-> validation and diagnosis
```

Modern NLP adds two further questions. First, how can the loop be executed when the model, optimizer states, and activations do not fit on one accelerator? Second, after a model can generate fluent text, how can its response distribution be adapted toward demonstrations or human preferences without confusing a proxy score with the real goal?

| Layer of the training problem | Main question | Typical methods |
|---|---|---|
| objective | What behavior receives credit? | cross-entropy, contrastive loss, preference loss |
| optimization | How are gradients converted into updates? | SGD, AdamW, schedules, clipping |
| systems | How is training made computationally feasible? | AMP, checkpointing, DDP, FSDP, parallelism |
| adaptation | Which knowledge and parameters should change? | continued pretraining, SFT, full fine-tuning, PEFT |
| post-training | How are preferred responses distinguished? | reward modeling, PPO, DPO |

This chapter follows that sequence. It first explains ordinary supervised and self-supervised training, then moves from single-device optimization to scalable training, task adaptation, parameter-efficient methods, and preference-based post-training.

### **What Does Training Mean in NLP?** {#what-does-training-mean-in-nlp}

In NLP, training means adjusting model parameters so that correct language outputs receive higher scores or probabilities than incorrect outputs.

For classification, the model learns:

$$
P(y \mid x)
$$

This means: given an input text $x$, estimate the probability of each possible label $y$.

| Symbol | Meaning | Example |
|---|---|---|
| $x$ | input text | `The movie was excellent` |
| $y$ | output label | `positive` |
| $P(y \mid x)$ | probability of label $y$ given input $x$ | `P(positive | text) = 0.96` |

For generation, the model learns:

$$
P(y_1, y_2, ..., y_T \mid x)
$$

Here the output is not one label. It is a sequence of tokens. The model must decide what token comes first, what comes next, and when to stop.

Training works because the model receives a feedback signal. The feedback signal is the **loss**. If the model predicts the wrong label or assigns low probability to the correct token, the loss becomes large. The optimizer then updates the parameters to reduce this loss on similar future examples. This is why a model can improve without being given explicit rules such as "excellent means positive" or "Google is often an organization". The rules are not written manually; they are gradually encoded in the learned parameters.

A single training step can be understood as:

```text
1. Read a mini-batch of NLP examples
2. Run the forward pass to produce logits or probabilities
3. Compare predictions with gold labels using a loss function
4. Run the backward pass to compute gradients
5. Update model parameters with an optimizer
```

The model improves only if these parts are aligned. A good architecture with the wrong objective may learn the wrong behavior. A good objective with poor optimization may fail to converge.

Suppose we train a small sentiment classifier.

```text
Input:  "The movie was excellent"
Target: positive
```

At the beginning:

```text
P(positive) = 0.51
P(negative) = 0.49
```

After training:

```text
P(positive) = 0.96
P(negative) = 0.04
```

<details>
<summary>Python One Training Step for Text Classification</summary>

```python
import torch
import torch.nn as nn

class TinyTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids):
        # Step 1: map token ids to dense vectors
        embeddings = self.embedding(input_ids)

        # Step 2: average token vectors into a sentence representation
        sentence_vector = embeddings.mean(dim=1)

        # Step 3: produce raw class scores
        logits = self.classifier(sentence_vector)
        return logits

model = TinyTextClassifier(vocab_size=10000, embed_dim=64, num_classes=2)
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

input_ids = torch.tensor([
    [101, 2023, 3185, 2003, 6581, 102],
    [101, 2023, 3185, 2003, 1175, 102]
])
labels = torch.tensor([1, 0])  # 1 = positive, 0 = negative

optimizer.zero_grad()       # clear old gradients
logits = model(input_ids)   # forward pass
loss = loss_fn(logits, labels)
loss.backward()             # backward pass
optimizer.step()            # parameter update

print("loss:", loss.item())
```
</details>

| Concept | Role | Intuition |
|---|---|---|
| Model | produces predictions | the function being trained |
| Objective | defines what should be learned | task-level learning goal |
| Loss | measures error | numerical feedback |
| Gradient | tells how parameters affect loss | direction signal |
| Optimizer | updates parameters | learning rule |

### **Training Objectives in NLP** {#training-objectives-in-nlp}

A training objective defines what the model is asked to learn. This is different from the model architecture. A Transformer can be trained as a classifier, a masked language model, a causal language model, or a sequence-to-sequence generator. The architecture controls how information can flow; the objective controls what learning signal the model receives.

| Objective | What the model learns | Example |
|---|---|---|
| Supervised learning | map input to labelled output | sentiment classification |
| Language modeling | predict next token | GPT-style pretraining |
| Masked language modeling | recover hidden tokens | BERT-style pretraining |
| Sequence-to-sequence training | generate output from input | translation |
| Contrastive learning | compare representations | retrieval embeddings |

#### **Supervised Learning** {#supervised-learning}

Supervised learning uses examples that already have correct answers. This is the most direct training setup: the dataset tells the model what output should be produced for each input.

```text
Input:  "The plot was boring"
Label:  Negative
```

The dataset can be written as:

$$
\{(x_i, y_i)\}_{i=1}^{N}
$$

| Symbol | Meaning |
|---|---|
| $N$ | number of training examples |
| $x_i$ | input text of the $i$-th example |
| $y_i$ | gold label of the $i$-th example |

The model predicts $P(y_i \mid x_i)$ for each example. Training increases the probability of the correct label and decreases the probability of incorrect labels.

For sentiment classification:

```text
Text: "boring plot"
Gold label: negative
Bad model:  P(positive)=0.70, P(negative)=0.30
Good model: P(positive)=0.05, P(negative)=0.95
```

<details>
<summary>Python Supervised Text Classification</summary>

```python
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline

texts = [
    "excellent movie with strong acting",
    "wonderful story and beautiful ending",
    "boring plot and weak characters",
    "terrible pacing and dull dialogue"
]
labels = [1, 1, 0, 0]

# Step 1: TF-IDF converts text into numeric features.
# Step 2: Logistic Regression learns from labelled examples.
model = Pipeline([
    ("tfidf", TfidfVectorizer()),
    ("classifier", LogisticRegression())
])

model.fit(texts, labels)
print(model.predict(["excellent story"]))
```
</details>

| Strength | Limitation |
|---|---|
| Directly learns the target task | needs labelled data |
| Simple to evaluate | labels may be expensive |
| strong for classification | may overfit small datasets |

#### **Causal Language Modeling** {#causal-language-modeling}
Language modeling trains a model to predict the next token from previous tokens. It is simple as a task definition, but extremely powerful because every piece of raw text can be converted into many next-token prediction examples.

```text
Input:  Natural language processing is
Target: useful
```

For a sequence $w_1, ..., w_T$, the probability is factorized as:

$$
P(w_1, ..., w_T)=\prod_{t=1}^{T}P(w_t \mid w_{<t})
$$

| Symbol | Meaning |
|---|---|
| $w_t$ | token at position $t$ |
| $w_{<t}$ | all tokens before position $t$ |
| $P(w_t \mid w_{<t})$ | probability of the next token given previous context |
| $\prod$ | multiply probabilities across positions |

This objective is powerful because raw text automatically provides labels. Every token is a target for the previous context.

<details>
<summary>Python Create Causal LM Inputs and Labels</summary>

```python
import torch

# Token IDs for: Natural language processing is useful
tokens = torch.tensor([[10, 25, 37, 8, 91]])

# Input excludes the final token.
input_ids = tokens[:, :-1]

# Labels exclude the first token.
# Each position predicts the next token.
labels = tokens[:, 1:]

print("input_ids:", input_ids)
print("labels:", labels)
```
</details>

| Aspect | Language Modeling |
|---|---|
| Label source | raw text itself |
| Typical architecture | decoder-only Transformer |
| Main strength | generation and scalable pretraining |
| Main limitation | next-token prediction does not guarantee truthfulness |

#### **Masked Language Modeling** {#masked-language-modeling}

Masked Language Modeling trains a model to recover hidden tokens from surrounding context. The key difference from causal language modeling is that the model can use both left and right context, so it is especially useful for learning representations for understanding tasks.

```text
Original: Natural language processing is useful
Masked:   Natural language [MASK] is useful
Target:   processing
```

Unlike causal language modeling, masked language modeling can use both left and right context.

$$
P(w_{masked} \mid w_{left}, w_{right})
$$

| Symbol | Meaning |
|---|---|
| $w_{masked}$ | original token hidden by `[MASK]` |
| $w_{left}$ | tokens before the masked position |
| $w_{right}$ | tokens after the masked position |

> Image for masked language modeling:
>
> ![BERT masked language modeling workflow](https://media.geeksforgeeks.org/wp-content/uploads/20231004220634/Bert-language-model-2.png)
>
> Source: [GeeksforGeeks - Understanding BERT NLP](https://www.geeksforgeeks.org/understanding-bert-nlp/)

<details>
<summary>Python Simple Masked Token Construction</summary>

```python
sentence = ["Natural", "language", "processing", "is", "useful"]
mask_index = 2

# Step 1: save the original token as target
target = sentence[mask_index]

# Step 2: replace the token with [MASK] in the input
masked_sentence = sentence.copy()
masked_sentence[mask_index] = "[MASK]"

print("input:", masked_sentence)
print("target:", target)
```
</details>

| Aspect | Causal LM | Masked LM |
|---|---|---|
| Context | left context only | left and right context |
| Typical model | GPT-style decoder | BERT-style encoder |
| Best for | generation | understanding |

#### **Sequence-to-Sequence Training** {#sequence-to-sequence-training}

Sequence-to-sequence training teaches a model to generate an output sequence conditioned on an input sequence. It is used when the answer cannot be represented as one label or one token, but must be produced as a full sequence.

```text
Source: I love NLP
Target: J'aime le TAL
```

The probability is:

$$
P(y \mid x)=\prod_{t=1}^{T}P(y_t \mid y_{<t}, x)
$$

| Symbol | Meaning |
|---|---|
| $x$ | source sequence |
| $y$ | target sequence |
| $y_t$ | target token at step $t$ |
| $y_{<t}$ | previous target tokens |

During training, seq2seq models often use **teacher forcing**: the decoder receives the correct previous token instead of its own previous prediction.

<details>
<summary>Python Teacher Forcing Inputs</summary>

```python
target = ["<bos>", "J'aime", "le", "TAL", "<eos>"]

# Decoder input is what the decoder receives.
decoder_input = target[:-1]

# Labels are what the decoder should predict.
labels = target[1:]

print("decoder_input:", decoder_input)
print("labels:", labels)
```
</details>

| Objective | Input | Output | Example |
|---|---|---|---|
| Classification | text | label | sentiment |
| Causal LM | prefix | next token | generation |
| Seq2Seq | source sequence | target sequence | translation |

#### **Contrastive Learning** {#contrastive-learning}

Contrastive learning trains representations by comparing examples. Related texts should be close in vector space, and unrelated texts should be far apart. Unlike classification, the goal is not always to predict a fixed label; often the goal is to build an embedding space that supports search, retrieval, or matching.

```text
Query:     How do I reset my password?
Positive:  I forgot my password, how can I change it?
Negative:  What is the weather tomorrow?
```

The model learns an embedding space. In that space, semantic similarity should correspond to geometric closeness. This is especially useful for retrieval, where a query vector is compared with many document vectors.

<details>
<summary>Python Contrastive Similarity Sketch</summary>

```python
import torch
import torch.nn.functional as F

query = torch.tensor([[1.0, 0.2, 0.1]])
positive = torch.tensor([[0.9, 0.25, 0.1]])
negative = torch.tensor([[0.1, 0.8, 0.4]])

# Step 1: compute cosine similarity
pos_sim = F.cosine_similarity(query, positive)
neg_sim = F.cosine_similarity(query, negative)

# Step 2: a good embedding model should make pos_sim > neg_sim
print("positive similarity:", pos_sim.item())
print("negative similarity:", neg_sim.item())
```
</details>

| Objective | Learns from | Best for |
|---|---|---|
| Supervised learning | labelled examples | classification |
| Causal LM | raw token sequences | generation |
| Masked LM | corrupted text | understanding |
| Seq2Seq | paired sequences | translation/summarisation |
| Contrastive learning | positive and negative pairs | retrieval/embedding |

### **Loss Functions** {#loss-functions}

#### **Cross-Entropy and Negative Log-Likelihood** {#cross-entropy-and-negative-log-likelihood}

Cross-entropy and negative log-likelihood are two views of the same core objective in ordinary softmax classification. Cross-entropy describes the discrepancy between a target distribution and a predicted distribution; negative log-likelihood describes the penalty for assigning low probability to the observed target. Keeping both interpretations is useful because the first emphasizes distributions and the second generalizes naturally to sequences and structured probabilistic models.

The two terms should not be treated as unrelated losses. With a one-hot target and a softmax model, cross-entropy reduces exactly to the negative logarithm of the probability assigned to the gold class.

Cross-entropy is the standard loss for classification and token prediction. It measures how surprised the model is by the correct answer. If the model is confident in the correct answer, the loss is low. If the model assigns very little probability to the correct answer, the loss becomes large.

For one example with correct class $y$, the loss is:

$$
L = -\log P(y \mid x)
$$

| Symbol | Meaning | NLP Example |
|---|---|---|
| $L$ | loss value | error for one sentence |
| $x$ | input text | `The movie was boring` |
| $y$ | correct label | `negative` |
| $P(y \mid x)$ | probability assigned to the correct label | `P(negative)=0.80` |

If the model gives high probability to the correct class, the loss is small. If it gives low probability, the loss is large.

| Correct class probability | Loss $-\log p$ | Meaning |
|---:|---:|---|
| 0.90 | 0.105 | confident and mostly correct |
| 0.50 | 0.693 | uncertain |
| 0.10 | 2.303 | very wrong |

<details>
<summary>Python Cross-Entropy Loss</summary>

```python
import torch
import torch.nn as nn

logits = torch.tensor([
    [2.0, 0.3, -1.2],
    [0.1, 1.7, 0.2]
])
labels = torch.tensor([0, 1])

loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, labels)

print(loss.item())
```
</details>

| If correct probability is... | Cross-entropy loss is... |
|---|---|
| high | low |
| low | high |
| zero or near zero | extremely high |

Negative Log-Likelihood (NLL) is the negative log probability assigned to the correct output. It is another way to express the same training intuition: maximize the probability of the correct answer, or equivalently minimize the negative log of that probability.

$$
L = -\log P(\text{correct output})
$$

NLL and cross-entropy are closely related. In many classification settings, cross-entropy is NLL applied after a softmax probability distribution.

```text
logits -> softmax -> probabilities -> negative log of correct probability
```

<details>
<summary>Python NLL Loss after Log Softmax</summary>

```python
import torch
import torch.nn as nn

logits = torch.tensor([[2.0, 0.3, -1.2]])
label = torch.tensor([0])

# Step 1: convert logits to log-probabilities
log_probs = torch.log_softmax(logits, dim=-1)

# Step 2: NLLLoss takes log-probabilities
loss_fn = nn.NLLLoss()
loss = loss_fn(log_probs, label)

print(loss.item())
```
</details>

| Loss | Input expected by PyTorch | Common use |
|---|---|---|
| `CrossEntropyLoss` | raw logits | most classification tasks |
| `NLLLoss` | log probabilities | when log-softmax is computed separately |



#### **Token-Level Loss vs Sequence-Level Loss** {#token-level-loss-vs-sequence-level-loss}

Token-level loss computes error at each token position. Sequence-level loss treats the whole generated sequence as the object being evaluated. Most neural NLP training uses token-level loss because it gives dense feedback at every position, while sequence-level objectives are harder to optimize directly.

For language modeling:

```text
Input:   I     love   NLP
Target:  love  NLP    <eos>
Loss:    L1    L2     L3
```

The final loss is usually averaged over non-padding tokens.

<details>
<summary>Python Ignore Padding in Token Loss</summary>

```python
import torch
import torch.nn as nn

logits = torch.randn(2, 4, 1000)  # [batch, seq_len, vocab_size]
labels = torch.tensor([
    [10, 25, 37, 2],
    [19, 44, 0, -100]
])

# ignore_index=-100 means this position is ignored in the loss.
loss_fn = nn.CrossEntropyLoss(ignore_index=-100)
loss = loss_fn(logits.view(-1, 1000), labels.view(-1))

print(loss.item())
```
</details>

| Loss Type | Unit | Typical Task |
|---|---|---|
| Token-level loss | each token | LM, NER, seq2seq training |
| Sequence-level score | whole sequence | decoding, evaluation, RL-style training |

#### **Class Imbalance and Weighted Loss** {#class-imbalance-and-weighted-loss}

Class imbalance happens when some labels appear much more often than others. In NLP, this is common in NER because most tokens are `O`. A model can appear accurate by predicting the majority label too often, even though it fails on the labels we actually care about.

If 95% of tokens are `O`, a model can get high accuracy by predicting `O` too often. Weighted loss gives rare classes more importance.

<details>
<summary>Python Weighted Cross-Entropy</summary>

```python
import torch
import torch.nn as nn

# Suppose class 0 is common and class 1 is rare.
class_weights = torch.tensor([1.0, 5.0])

loss_fn = nn.CrossEntropyLoss(weight=class_weights)

logits = torch.tensor([[1.2, 0.4], [0.3, 1.8]])
labels = torch.tensor([0, 1])

loss = loss_fn(logits, labels)
print(loss.item())
```
</details>

| Problem | Solution |
|---|---|
| majority label dominates | weighted loss |
| rare labels ignored | oversampling / better metrics |
| accuracy looks high but model is bad | use precision, recall, F1 |

### **Backpropagation and Gradient-Based Learning** {#backpropagation-and-gradient-based-learning}

Backpropagation and gradient-based learning explain how a neural NLP model turns a loss value into parameter updates. The forward pass computes predictions. The loss measures error. The backward pass computes gradients. Gradient descent then uses those gradients to update the parameters.

The core idea is:

```text
prediction error -> gradients -> parameter update -> lower future error
```

Backpropagation itself is not the optimizer. It does not decide how large the update should be. Its job is to efficiently compute gradients. The optimizer then uses those gradients to update parameters.

> Image for backpropagation:
>
> ![Backpropagation neural network diagram](https://upload.wikimedia.org/wikipedia/commons/thumb/6/60/ArtificialNeuronModel_english.png/500px-ArtificialNeuronModel_english.png)
>
> Source: [Wikipedia - Backpropagation](https://en.wikipedia.org/wiki/Backpropagation)

#### **Forward Pass** {#forward-pass}

The forward pass is the computation from input to prediction. It is called "forward" because information moves from the input side of the model toward the output side.

For a neural NLP classifier, the process usually looks like this:

```text
text -> tokens -> token IDs -> embeddings -> model layers -> logits -> probabilities
```

For example:

```text
Input text:     "This movie is excellent"
Token IDs:      [101, 2023, 3185, 2003, 6581, 102]
Logits:         [-1.2, 2.8]
Probabilities:  negative = 0.018, positive = 0.982
```

The model first converts token IDs into embeddings. Then the architecture combines those embeddings. A simple model may average token embeddings. An RNN processes them sequentially. A Transformer uses self-attention to build contextual representations. Finally, a prediction head produces logits.

Logits are raw scores. They are not probabilities yet. For classification, softmax converts logits into probabilities:

$$
P(y=k \mid x)=\frac{e^{z_k}}{\sum_j e^{z_j}}
$$

| Symbol | Meaning |
|---|---|
| $z_k$ | logit score for class $k$ |
| $e^{z_k}$ | exponentiated positive score |
| $\sum_j e^{z_j}$ | sum over all candidate classes |
| $P(y=k \mid x)$ | probability of class $k$ |

No learning happens during the forward pass. The model is only using its current parameters to compute outputs. Learning starts after the loss compares these outputs with the correct answer.

<details>
<summary>Python Forward Pass with Comments</summary>

```python
import torch
import torch.nn as nn

class TinyTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids):
        # Step 1: token IDs -> embeddings
        embeddings = self.embedding(input_ids)

        # Step 2: combine token vectors into one sentence vector
        sentence_vector = embeddings.mean(dim=1)

        # Step 3: sentence vector -> raw class scores
        logits = self.classifier(sentence_vector)

        return logits

model = TinyTextClassifier(vocab_size=10000, embed_dim=64, num_classes=2)

input_ids = torch.tensor([[101, 2023, 3185, 2003, 6581, 102]])

# Forward pass: compute logits with current parameters
logits = model(input_ids)

# Convert logits to probabilities for interpretation
probs = torch.softmax(logits, dim=-1)

print("logits:", logits)
print("probabilities:", probs)
```
</details>

Summary:

| Item | Role |
|---|---|
| embeddings | numeric token representations |
| model layers | combine information |
| logits | raw prediction scores |
| probabilities | normalized scores |
| forward pass | computes predictions, does not update parameters |

#### **Backward Pass** {#backward-pass}

The backward pass computes gradients of the loss with respect to model parameters.

$$
\frac{\partial L}{\partial \theta}
$$

Here:

| Symbol | Meaning |
|---|---|
| $L$ | loss value |
| $\theta$ | model parameters |
| $\frac{\partial L}{\partial \theta}$ | how the loss changes when parameters change |

The backward pass starts from the loss and moves backward through the computation graph. It uses the chain rule to compute how each parameter contributed to the final error.

For a simple classifier:

```text
loss -> logits -> classifier weights -> embeddings -> earlier parameters
```

The important point is that the backward pass does not directly update the parameters. It fills the `.grad` field of each parameter. The optimizer uses those gradients later.

<details>
<summary>Python Backward Pass with Comments</summary>

```python
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)

input_ids = torch.tensor([
    [101, 2023, 3185, 2003, 6581, 102],
    [101, 2023, 3185, 2003, 1175, 102]
])
labels = torch.tensor([1, 0])

# Step 1: clear gradients from the previous batch
optimizer.zero_grad()

# Step 2: forward pass
logits = model(input_ids)

# Step 3: compute loss
loss = loss_fn(logits, labels)

# Step 4: backward pass computes gradients
loss.backward()

# At this point, parameters have gradients but have not been updated yet.
for name, param in model.named_parameters():
    if param.grad is not None:
        print(name, param.grad.norm().item())
```
</details>

Comparison:

| Pass | Direction | Output | Updates Parameters? |
|---|---|---|---|
| Forward pass | input to output | logits / probabilities / loss | no |
| Backward pass | loss to parameters | gradients | no |
| Optimizer step | uses gradients | new parameters | yes |

#### **Gradient Descent Intuition** {#gradient-descent-intuition}

Gradient descent is the idea of updating parameters in the direction that reduces loss. If the loss is like a landscape, training tries to move the parameters downhill.

The basic update rule is:

$$
\theta \leftarrow \theta - \eta \nabla_\theta L
$$

| Symbol | Meaning | Intuition |
|---|---|---|
| $\theta$ | parameters | current model weights |
| $L$ | loss | how wrong the model is |
| $\nabla_\theta L$ | gradient | direction of steepest loss increase |
| $\eta$ | learning rate | step size |

The gradient points toward increasing loss. Since training wants lower loss, the update subtracts the gradient.

> Image for gradient descent:
>
> ![Gradient descent](https://upload.wikimedia.org/wikipedia/commons/thumb/f/ff/Gradient_descent.svg/250px-Gradient_descent.svg.png)
>
> Source: [Wikipedia - Gradient descent](https://en.wikipedia.org/wiki/Gradient_descent)

Learning rate is crucial:

| Learning Rate | Behavior |
|---|---|
| too small | training is very slow |
| too large | training may jump around or diverge |
| reasonable | loss decreases steadily |

<details>
<summary>Python Manual Gradient Descent Example</summary>

```python
import torch

# A toy parameter. The loss is minimized when w = 3.
w = torch.tensor([1.0], requires_grad=True)

# Step 1: define a simple loss
loss = (w - 3) ** 2

# Step 2: compute gradient d(loss)/d(w)
loss.backward()

print("w before update:", w.item())
print("gradient:", w.grad.item())

# Step 3: manually apply gradient descent
learning_rate = 0.1
with torch.no_grad():
    w -= learning_rate * w.grad

print("w after update:", w.item())
```
</details>

For neural NLP models, we usually do not manually update parameters like this. Optimizers such as SGD, Adam, or AdamW perform the update. But the core idea remains the same: use gradients to reduce loss.

#### **Backpropagation Through Time** {#backpropagation-through-time}

Backpropagation Through Time (BPTT) is backpropagation applied to recurrent models unfolded across time steps.

An RNN reuses the same cell at every token position:

```text
x1 -> h1 -> h2 -> h3 -> h4
```

To train the RNN, the model is conceptually unfolded into a deep computation graph over time. The loss at later time steps must send gradients backward through earlier hidden states.

> Image for BPTT:
>
> ![Backpropagation through time unfolding](https://upload.wikimedia.org/wikipedia/commons/thumb/e/ee/Unfold_through_time.png/500px-Unfold_through_time.png)
>
> Source: [Wikipedia - Backpropagation through time](https://en.wikipedia.org/wiki/Backpropagation_through_time)

This matters in NLP because many sequences are long. If gradients must pass through many time steps, two problems may appear:

| Problem | Meaning | Effect |
|---|---|---|
| vanishing gradient | gradients become tiny | early tokens learn weakly |
| exploding gradient | gradients become huge | training becomes unstable |

This is one reason LSTM and GRU were introduced. Their gates help information and gradients survive over longer spans. Gradient clipping is also commonly used to control exploding gradients.

<details>
<summary>Python RNN Backward Pass Sketch</summary>

```python
import torch
import torch.nn as nn

rnn = nn.RNN(input_size=16, hidden_size=32, batch_first=True)
classifier = nn.Linear(32, 2)
loss_fn = nn.CrossEntropyLoss()

# batch of 4 sequences, each with 10 time steps
x = torch.randn(4, 10, 16)
labels = torch.tensor([0, 1, 0, 1])

# Step 1: forward through all time steps
outputs, final_hidden = rnn(x)

# Step 2: classify using final hidden state
logits = classifier(final_hidden[-1])

# Step 3: compute sequence-level classification loss
loss = loss_fn(logits, labels)

# Step 4: gradients flow backward through time
loss.backward()
```
</details>

Summary:

| Concept | Main Role | NLP Example |
|---|---|---|
| Forward pass | compute predictions | classify sentence sentiment |
| Backward pass | compute gradients | find how weights affected error |
| Gradient descent | update parameters | reduce future loss |
| BPTT | backprop through sequence steps | train RNN/LSTM/GRU |

### **Optimization Algorithms** {#optimization-algorithms}

Optimization algorithms decide how a model's parameters should move after gradients have been computed. Backpropagation tells us the direction and sensitivity of the loss with respect to each parameter. The optimizer turns that gradient information into an actual update step.

In NLP, optimization is especially important because models often have many parameters, sparse token signals, long sequences, and unstable early training dynamics. A good optimizer does not change the training objective. It changes how efficiently and stably the model moves toward a lower-loss region.

The basic optimization loop is:

```text
mini-batch -> forward pass -> loss -> backward pass -> gradients -> optimizer step
```

The optimizer mainly answers three questions:

| Question | Meaning |
|---|---|
| Which direction should parameters move? | usually opposite to the gradient |
| How large should the step be? | controlled by learning rate and adaptive scaling |
| How should past gradients affect the current step? | controlled by momentum or moment estimates |

#### **Stochastic Gradient Descent** {#stochastic-gradient-descent}

Stochastic Gradient Descent, usually abbreviated as SGD, is the most direct optimizer. It updates parameters by moving them a small distance in the opposite direction of the gradient.

The update rule is:

$$
\theta_{t+1} = \theta_t - \eta g_t
$$

Here, $\theta_t$ means the model parameters at training step $t$. The gradient $g_t = \nabla_\theta L(\theta_t)$ tells us how the loss changes if each parameter changes. The learning rate $\eta$ controls the step size. If $\eta$ is large, the model moves aggressively. If $\eta$ is small, the model moves cautiously.

The word "stochastic" means the gradient is usually estimated from one mini-batch rather than the entire dataset. For example, when training a sentiment classifier, one update may use 32 reviews instead of all reviews. This makes training much faster, but the gradient becomes noisy because each mini-batch only gives a partial view of the full objective.

The process looks like this:

```text
batch 1 -> noisy gradient -> update
batch 2 -> noisy gradient -> update
batch 3 -> noisy gradient -> update
...
```

This noise is not always bad. It can help the model escape very sharp or poor local regions. However, it also means SGD may zig-zag, especially when the loss surface has narrow valleys.

A small PyTorch example shows the basic workflow:

<details>
<summary>Python SGD Example</summary>

```python
import torch
import torch.nn as nn

# A tiny text classifier: embedding -> mean pooling -> linear classifier
class TinyTextClassifier(nn.Module):
    def __init__(self, vocab_size=1000, embed_dim=16, num_classes=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.classifier = nn.Linear(embed_dim, num_classes)

    def forward(self, input_ids):
        # Step 1: convert token IDs into dense vectors
        embeddings = self.embedding(input_ids)

        # Step 2: average token embeddings into one sentence vector
        sentence_vector = embeddings.mean(dim=1)

        # Step 3: produce class logits
        logits = self.classifier(sentence_vector)
        return logits

model = TinyTextClassifier()
loss_fn = nn.CrossEntropyLoss()

# SGD uses the raw gradient direction and one global learning rate.
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)

input_ids = torch.tensor([
    [12, 45, 98, 31],
    [10, 22, 76, 88]
])
labels = torch.tensor([1, 0])

optimizer.zero_grad()          # clear gradients from the previous step
logits = model(input_ids)      # forward pass
loss = loss_fn(logits, labels) # compute classification loss
loss.backward()                # compute gradients
optimizer.step()               # theta <- theta - learning_rate * gradient

print(float(loss))
```

</details>

SGD is easy to understand and has low memory cost because it does not need to store many extra statistics. But for modern NLP models, especially Transformers, plain SGD often requires careful learning-rate tuning and may converge slowly.

#### **Momentum** {#momentum}

Momentum improves SGD by remembering the recent direction of movement. Instead of using only the current gradient, it keeps a running velocity. This velocity accumulates directions that appear consistently across steps and weakens directions that keep changing.

The update can be written as:

$$
v_t = \beta v_{t-1} + g_t
$$

$$
\theta_{t+1} = \theta_t - \eta v_t
$$

Here, $v_t$ is the velocity at step $t$. The coefficient $\beta$ is usually a value such as $0.9$. A larger $\beta$ means the optimizer remembers more of the past direction. The term $g_t$ is the current gradient.

A useful intuition is to imagine optimization as moving down a curved valley. Plain SGD may bounce from side to side because each mini-batch gradient is noisy. Momentum smooths these movements. If many gradients point roughly forward, the velocity grows in that direction. If gradients alternate left and right, they partially cancel out.

```text
SGD:       current gradient decides most of the step
Momentum:  past direction + current gradient decide the step
```

Momentum is useful in NLP when gradients are noisy, such as training RNNs or smaller neural text classifiers from scratch. It is less common than AdamW for fine-tuning large pretrained Transformers, but the idea of momentum is still important because Adam also uses a momentum-like first moment.

<details>
<summary>Python Momentum Example</summary>

```python
# Momentum is added as one argument to SGD.
# momentum=0.9 means the update keeps a strong memory of previous gradients.
optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.05,
    momentum=0.9
)

for input_ids, labels in dataloader:
    optimizer.zero_grad()
    logits = model(input_ids)
    loss = loss_fn(logits, labels)
    loss.backward()
    optimizer.step()
```

</details>

Compared with plain SGD, momentum usually reaches useful regions faster and follows a smoother path. The trade-off is that it adds another hyperparameter and can overshoot if the learning rate is too high.

#### **Adam** {#adam}

Adam stands for Adaptive Moment Estimation. It combines two ideas: momentum and adaptive learning rates. Instead of using the same effective step size for every parameter, Adam adjusts each parameter based on the recent behavior of its gradients.

Adam keeps two moving averages:

| Quantity | Meaning | Intuition |
|---|---|---|
| first moment $m_t$ | moving average of gradients | direction, similar to momentum |
| second moment $v_t$ | moving average of squared gradients | scale, how large gradients usually are |

The main equations are:

$$
m_t = \beta_1 m_{t-1} + (1 - \beta_1)g_t
$$

$$
v_t = \beta_2 v_{t-1} + (1 - \beta_2)g_t^2
$$

$$
\hat{m}_t = \frac{m_t}{1 - \beta_1^t}, \qquad
\hat{v}_t = \frac{v_t}{1 - \beta_2^t}
$$

$$
\theta_{t+1} = \theta_t - \eta \frac{\hat{m}_t}{\sqrt{\hat{v}_t} + \epsilon}
$$

The first equation builds a smoothed gradient direction. The second equation tracks the typical squared gradient size for each parameter. The bias-corrected terms $\hat{m}_t$ and $\hat{v}_t$ are needed because both moving averages start at zero. Without correction, the early estimates would be too small. The small value $\epsilon$ prevents division by zero.

In NLP, Adam is helpful because different parameters may receive very different gradient patterns. Word embeddings for frequent tokens may be updated often. Embeddings for rare tokens may receive sparse updates. Attention and feed-forward layers may also have different gradient scales. Adam handles this by giving each parameter its own adaptive update scale.

A simplified interpretation is:

```text
Adam step = direction from momentum / estimated gradient scale
```

<details>
<summary>Python Adam Example</summary>

```python
# Adam is often a strong default when training neural NLP models from scratch.
optimizer = torch.optim.Adam(
    model.parameters(),
    lr=1e-3,
    betas=(0.9, 0.999), # beta1 for first moment, beta2 for second moment
    eps=1e-8            # numerical stability term
)

for input_ids, labels in dataloader:
    optimizer.zero_grad()

    # Forward pass: produce predictions from token IDs
    logits = model(input_ids)

    # Loss: compare predictions with gold labels
    loss = loss_fn(logits, labels)

    # Backward pass: compute gradients
    loss.backward()

    # Adam step: use first and second moment estimates to update parameters
    optimizer.step()
```

</details>

Adam is usually faster and easier to tune than SGD, but it can sometimes generalize differently from SGD. It also uses more memory because it stores moment estimates for every trainable parameter.

#### **AdamW** {#adamw}

AdamW is a variant of Adam designed to handle weight decay more cleanly. Weight decay is a regularization technique that discourages parameters from becoming too large. In ordinary Adam, weight decay can become entangled with the adaptive gradient update. AdamW decouples weight decay from the gradient-based update.

The key difference is:

```text
Adam:  gradient update and weight decay are coupled
AdamW: gradient update and weight decay are applied separately
```

This matters for Transformer models because fine-tuning usually starts from a pretrained model. We do not want the optimizer to distort pretrained weights too aggressively, but we still want regularization. AdamW gives a more predictable way to apply that regularization.

A practical AdamW update can be understood in two conceptual steps:

```text
Step 1: use Adam-style moments to update parameters from gradients
Step 2: apply weight decay directly to parameters
```

In modern NLP, AdamW is commonly used for BERT-style fine-tuning, GPT-style language-model training, and many encoder-decoder models. Typical fine-tuning learning rates are small, often around `1e-5` to `5e-5`, because pretrained models already contain useful representations.

<details>
<summary>Python AdamW Fine-Tuning Example</summary>

```python
from torch.optim import AdamW

# AdamW is the common choice for Transformer fine-tuning.
# lr is small because the pretrained model should be adjusted carefully.
# weight_decay regularizes weights without mixing decay into Adam's moments.
optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

for batch in dataloader:
    optimizer.zero_grad()

    # Transformer models often return the loss directly when labels are given.
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()
```

</details>

AdamW is not automatically better for every problem, but it is usually the safest first choice for pretrained Transformer-based NLP workflows.

#### **Learning Rate Scheduling** {#learning-rate-scheduling}

The learning rate controls how large each optimization step is. Even with a good optimizer, a poor learning rate can ruin training. If the learning rate is too high, the model may jump over useful regions and the loss may explode. If it is too low, the model may improve very slowly or get stuck before reaching a good solution.

A learning rate scheduler changes the learning rate during training. Instead of using one fixed value, the schedule controls the training rhythm.

Common schedules include:

| Schedule | Pattern | Typical use |
|---|---|---|
| constant | same learning rate throughout training | simple baselines |
| step decay | reduce learning rate at fixed milestones | classical neural training |
| linear decay | gradually reduce to zero | Transformer fine-tuning |
| cosine decay | smooth decrease with cosine curve | large-scale pretraining |
| warmup + decay | increase first, then decrease | Transformer training |

Warmup is especially common in Transformer training. At the beginning, the model head may be randomly initialized, gradients can be unstable, and Adam-style moment estimates are still unreliable. Warmup starts with a very small learning rate and gradually increases it. After warmup, decay slowly reduces the learning rate so later training makes smaller, more precise updates.

```text
early training:      small learning rate for stability
middle training:     larger learning rate for fast learning
late training:       smaller learning rate for refinement
```

For fine-tuning pretrained NLP models, a linear warmup plus linear decay schedule is a common practical choice.

<details>
<summary>Python AdamW with Linear Warmup and Decay</summary>

```python
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup

num_epochs = 3
num_training_steps = num_epochs * len(dataloader)
num_warmup_steps = int(0.1 * num_training_steps)

optimizer = AdamW(
    model.parameters(),
    lr=2e-5,
    weight_decay=0.01
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

for epoch in range(num_epochs):
    for batch in dataloader:
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss.backward()

        # Step 1: update parameters using AdamW
        optimizer.step()

        # Step 2: update learning rate according to the schedule
        scheduler.step()
```

</details>

A useful practical rule is: choose the optimizer and scheduler together. AdamW with warmup and decay is a strong default for Transformer fine-tuning. SGD or SGD with momentum can still be useful for simpler models, smaller networks, and cases where memory cost matters.

> Image for optimizer path comparison:
>
> ![Optimization paths of SGD, Momentum, RMSProp, Adam, and related optimizers](https://data.aimsciences.org/aimsmath-data/jimo/2024/7/PIC/1547-5816_2024_7_2516-1.jpg)
>
> This figure compares the update paths of several optimizers on the same quadratic loss surface, including SGD, Momentum, RMSProp, Adam, and related variants. It is a better match for this section because it shows both the classical gradient-based family and adaptive optimizers. AdamW is not usually shown as a separate path in this kind of figure because its main difference from Adam is decoupled weight decay rather than a completely different search trajectory.
>
> Source: [A modification of adaptive moment estimation (adam) for machine learning - Figure 1](https://www.aimsciences.org/article/doi/10.3934/jimo.2024014)

| Optimizer | Main idea | Strength | Weakness | Common NLP use |
|---|---|---|---|---|
| SGD | move opposite the mini-batch gradient | simple, low memory | sensitive to learning rate, can be slow | simple neural baselines |
| Momentum | smooth SGD with past gradients | faster and less zig-zag | extra hyperparameter, can overshoot | RNN/CNN training from scratch |
| Adam | adaptive per-parameter learning rates | fast, handles sparse/noisy gradients | more memory, may need regularization care | neural NLP training from scratch |
| AdamW | Adam with decoupled weight decay | stable for pretrained Transformers | still needs careful learning rate | BERT/GPT fine-tuning |
| Scheduler | changes learning rate over time | improves stability and convergence | adds schedule choices | warmup + decay for Transformers |

### **Training Neural NLP Models** {#training-neural-nlp-models}

The loss and optimizer define the mathematical update, but a real NLP batch also contains variable-length sequences, padding, masks, decoder inputs, and occasionally unstable gradients. These details are part of the objective: an incorrect loss mask changes which tokens the model is rewarded for predicting.

This section focuses on the mechanics required even on one device. Techniques whose main purpose is memory reduction or multi-device scaling are separated into the next section so that correctness and scalability do not become mixed together.

| Mechanism | What it controls | Common failure |
|---|---|---|
| mini-batching | gradient estimate and throughput | noisy or memory-heavy updates |
| padding and masks | valid sequence positions | learning from `<pad>` tokens |
| teacher forcing | decoder history during training | train-inference mismatch |
| gradient clipping | maximum update signal | exploding gradients and `NaN` loss |

#### **Mini-Batch Training** {#mini-batch-training}

Mini-batch training means updating the model from a small group of examples at each step instead of using one example or the entire dataset. In NLP, a mini-batch may contain sentences, documents, dialogue turns, or source-target pairs for translation.

The reason mini-batches are used is both statistical and computational. Statistically, a mini-batch gives a more stable gradient estimate than a single example. Computationally, GPUs and TPUs are designed to process many examples in parallel. If we train one sentence at a time, most of the hardware sits idle.

There are three common training modes:

| Mode | Gradient Source | Strength | Weakness |
|---|---|---|---|
| single-example SGD | one example | very frequent updates | noisy, inefficient on GPU |
| mini-batch training | small batch | good balance of speed and gradient quality | needs padding for variable lengths |
| full-batch training | entire dataset | stable gradient | too slow and memory-heavy for neural NLP |

For NLP, the difficulty is that examples have different lengths:

```text
Sentence A: I love NLP
Sentence B: This course is very useful
Sentence C: Transformers are powerful but expensive
```

A tensor batch needs rectangular shape, but natural language is not rectangular. This is why batching is usually connected to padding, attention masks, and sometimes length-based batching.

A typical PyTorch `collate_fn` builds a batch by padding examples to the longest sequence inside that batch:

<details>
<summary>Python Mini-Batch Collation Example</summary>

```python
import torch
from torch.nn.utils.rnn import pad_sequence

PAD_ID = 0

def collate_text_classification(batch):
    """
    batch is a list of examples.
    Each example contains token IDs and a label.

    Example item:
    {
        "input_ids": tensor([101, 2023, 2003, 2204, 102]),
        "label": 1
    }
    """
    input_ids = [item["input_ids"] for item in batch]
    labels = torch.tensor([item["label"] for item in batch], dtype=torch.long)

    # Step 1: pad shorter sequences so every row has the same length.
    padded_input_ids = pad_sequence(
        input_ids,
        batch_first=True,
        padding_value=PAD_ID
    )

    # Step 2: mark real tokens as 1 and padding positions as 0.
    attention_mask = (padded_input_ids != PAD_ID).long()

    return {
        "input_ids": padded_input_ids,
        "attention_mask": attention_mask,
        "labels": labels
    }
```

</details>

Mini-batch size is not just a speed setting. It changes optimization behavior. Larger batches give smoother gradients, but they use more memory and may require a different learning rate. Smaller batches add noise, which can sometimes help generalization but may make training unstable.

A useful practical rule is to choose the largest batch size that fits memory comfortably, then tune the learning rate and gradient accumulation if needed.

#### **Padding and Attention Masks** {#padding-and-attention-masks}

Padding is the process of adding special `<pad>` tokens to make sequences in a batch have the same length. An attention mask tells the model which positions are real tokens and which positions are padding.

For example:

```text
Original sentences:
A: [101, 2023, 2003, 2204, 102]
B: [101, 2307, 102]

After padding:
A: [101, 2023, 2003, 2204, 102]
B: [101, 2307, 102,    0,   0]

Attention mask:
A: [1, 1, 1, 1, 1]
B: [1, 1, 1, 0, 0]
```

The model should learn from real tokens, not from artificial padding. Without attention masks, a Transformer may allow real tokens to attend to padding positions. That introduces meaningless context. For classification, it can corrupt the sentence representation. For token classification, it can cause the loss to train on fake token positions. For generation, it can make the decoder learn padding patterns instead of language patterns.

In Transformer attention, the simplified computation is:

$$
\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T + M}{\sqrt{d_k}}\right)V
$$

Here, $Q$, $K$, and $V$ are query, key, and value matrices. The term $d_k$ is the key/query dimension used for scaling. The mask $M$ modifies attention scores before softmax. Real-token positions receive normal scores, while padding positions receive a very negative value, so after softmax their probability becomes almost zero.

```text
before mask:  [2.1, 1.4, 0.7, 0.3]
after mask:   [2.1, 1.4, -inf, -inf]
after softmax: real tokens receive probability, padding tokens receive ~0
```

Padding also matters for the loss function. If a sequence-to-sequence label contains `<pad>` positions, those positions should usually be ignored:

<details>
<summary>Python Padding, Attention Mask, and Ignored Loss</summary>

```python
import torch
import torch.nn as nn

PAD_ID = 0
IGNORE_INDEX = -100

input_ids = torch.tensor([
    [101, 2023, 2003, 2204, 102],
    [101, 2307, 102,    0,   0]
])

attention_mask = (input_ids != PAD_ID).long()

# For token-level labels, padding positions should not contribute to the loss.
labels = torch.tensor([
    [5, 7, 7, 3, 2],
    [5, 3, 2, IGNORE_INDEX, IGNORE_INDEX]
])

logits = torch.randn(2, 5, 10)  # batch_size=2, seq_len=5, num_labels=10

loss_fn = nn.CrossEntropyLoss(ignore_index=IGNORE_INDEX)

# CrossEntropyLoss expects shape: [N, C]
loss = loss_fn(
    logits.view(-1, 10),
    labels.view(-1)
)

print("attention mask:")
print(attention_mask)
print("loss:", float(loss))
```

</details>

Padding strategy affects efficiency. Padding every batch to the longest sequence in the whole dataset wastes memory. Padding only to the longest sequence inside the current mini-batch is more efficient. Sorting or grouping examples by similar length can reduce padding even more.

#### **Teacher Forcing** {#teacher-forcing}

Teacher forcing is a training strategy for sequence-to-sequence models. During training, the decoder receives the correct previous target token instead of its own previous prediction.

For translation, suppose the target sentence is:

```text
<bos> J'aime le TAL <eos>
```

The decoder input and label are shifted versions of the same target:

```text
Decoder input: <bos> J'aime le TAL
Label:         J'aime le TAL <eos>
```

At each step, the decoder learns to predict the next token given the source sentence and the correct previous target tokens.

> Image for teacher forcing:
>
> ![Sequence-to-sequence encoder-decoder with teacher forcing](https://alvinntnu.github.io/NTNU_ENC2045_LECTURES/_images/seq2seq-enc-dec-1.gif)
>
> Source: [ENC2045 Computational Linguistics - Attention and Transformers: Intuitions](https://alvinntnu.github.io/NTNU_ENC2045_LECTURES/nlp/dl-attention-transformer-intuition.html)

Teacher forcing makes training easier because early decoder inputs are clean. If the model is still weak, it does not have to recover from its own previous mistakes during training. This gives a clearer learning signal.

The weakness is exposure bias. At inference time, the model does not receive the gold previous token. It receives its own generated token. If it makes one bad prediction, the next prediction is conditioned on a context that may be unlike anything it saw during teacher-forced training.

```text
training:  correct history -> predict next token
inference: model history   -> predict next token
```

This gap is one reason generation quality depends strongly on decoding strategy, scheduled sampling, reinforcement-style fine-tuning, or instruction tuning in larger systems.

<details>
<summary>Python Teacher Forcing for Seq2Seq Labels</summary>

```python
import torch

BOS_ID = 1
EOS_ID = 2
PAD_ID = 0
IGNORE_INDEX = -100

# Target token IDs for two examples, already padded.
# Example 1: <bos> J'aime le TAL <eos>
# Example 2: <bos> Salut <eos> <pad> <pad>
target_ids = torch.tensor([
    [BOS_ID, 42, 51, 63, EOS_ID],
    [BOS_ID, 77, EOS_ID, PAD_ID, PAD_ID]
])

# Decoder input excludes the final token.
# The decoder receives previous gold tokens.
decoder_input_ids = target_ids[:, :-1]

# Labels exclude the first token.
# The model is trained to predict the next token at each position.
labels = target_ids[:, 1:].clone()

# Padding positions should not contribute to the loss.
labels[labels == PAD_ID] = IGNORE_INDEX

print("decoder inputs:")
print(decoder_input_ids)
print("labels:")
print(labels)
```

</details>

For encoder-decoder models such as T5, BART, or many translation systems, this shifted-input pattern is central to training. For decoder-only language models, the same idea appears as next-token prediction: the input sequence predicts the sequence shifted one token to the left.

#### **Gradient Clipping** {#gradient-clipping}

Gradient clipping limits the size of gradients before the optimizer updates the model. It is a safeguard against exploding gradients.

Exploding gradients happen when the gradient norm becomes extremely large. In sequence models, this can occur because gradients are repeatedly multiplied through many time steps or many layers. In large Transformers, it can also appear during unstable early fine-tuning, bad learning-rate choices, or noisy batches.

The most common method is clipping by global norm:

$$
\text{if } \lVert g \rVert_2 > c, \quad g \leftarrow g \cdot \frac{c}{\lVert g \rVert_2}
$$

Here, $g$ is the collection of gradients across model parameters. The value $c$ is the maximum allowed norm, such as `1.0`. If the gradient norm is already small, nothing changes. If it is too large, the whole gradient vector is rescaled while preserving its direction.

```text
large gradient direction: keep direction
large gradient magnitude: shrink magnitude
```

This is different from simply lowering the learning rate. Lowering the learning rate shrinks every update, even normal ones. Gradient clipping only intervenes when a step is unusually large.

<details>
<summary>Python Training Loop with Gradient Clipping</summary>

```python
import torch
from torch.nn.utils import clip_grad_norm_

model.train()
max_grad_norm = 1.0

for batch in dataloader:
    optimizer.zero_grad()

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )

    loss = outputs.loss
    loss.backward()

    # Step 1: compute and clip the global gradient norm.
    # If gradients are too large, they are rescaled before optimizer.step().
    grad_norm = clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)

    # Step 2: update parameters with clipped gradients.
    optimizer.step()
    scheduler.step()

    print("loss:", float(loss), "gradient norm before clipping:", float(grad_norm))
```

</details>

Gradient clipping is especially common in RNN, LSTM, GRU, and Transformer training. It does not fix the root cause of instability, but it prevents a single bad batch from destroying training.

### **Efficient and Scalable Training** {#efficient-and-scalable-training}

Large-model training is limited by more than parameter count. Memory is consumed by parameters, gradients, optimizer states, saved activations, temporary attention buffers, and the input batch. Compute is consumed again during the backward pass, communication appears when states are distributed, and storage is needed for checkpoints. A scalable design therefore asks which state can be reduced, recomputed, sharded, or communicated.

For Adam-style training in full precision, each parameter can require storage for the parameter itself, its gradient, and two optimizer moments. Mixed precision may also keep an FP32 master copy. Activations depend on batch size and sequence length, and attention-related memory can grow quickly with context length.

| Resource pressure | First techniques to consider | What they trade |
|---|---|---|
| batch does not fit | shorter sequences, bucketing, gradient accumulation | more optimizer steps or latency |
| activations dominate | mixed precision, activation checkpointing | lower precision or recomputation |
| model states dominate | ZeRO/FSDP, tensor parallelism | communication and implementation complexity |
| one device is too slow | data parallelism | replicated model memory |
| one layer cannot fit | tensor/model parallelism | frequent collective communication |
| model depth creates idle devices | pipeline parallelism with micro-batches | pipeline bubbles and scheduling |

#### **Gradient Accumulation and Effective Batch Size** {#gradient-accumulation-and-effective-batch-size}

Gradient accumulation splits one logical batch into several micro-batches. Each micro-batch performs a forward and backward pass, but the optimizer updates parameters only after several gradients have been accumulated. It is useful when the desired batch is larger than device memory permits.

With data parallelism, the effective global batch size is:

$$
B_{\text{global}}
=B_{\text{micro}}\times K_{\text{accum}}\times D_{\text{data}}
$$

| Symbol | Meaning |
|---|---|
| $B_{\text{micro}}$ | examples processed by each device in one forward pass |
| $K_{\text{accum}}$ | micro-batches accumulated before an optimizer step |
| $D_{\text{data}}$ | number of data-parallel workers |
| $B_{\text{global}}$ | examples contributing to one parameter update |

If the framework returns the mean loss for each micro-batch, dividing by $K_{\text{accum}}$ before `backward()` makes the accumulated gradient match the mean over the logical batch. The scheduler, logging of update steps, gradient clipping, and exponential moving averages should advance on optimizer updates rather than every micro-batch.

<details>
<summary>Python Gradient Accumulation Loop</summary>

```python
import torch
from torch.nn.utils import clip_grad_norm_

accumulation_steps = 4
max_grad_norm = 1.0

model.train()
optimizer.zero_grad(set_to_none=True)

for micro_step, batch in enumerate(train_loader, start=1):
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )

    # The model returns a mean loss for this micro-batch.
    # Scale it so that K accumulated gradients represent one mean batch loss.
    loss = outputs.loss / accumulation_steps
    loss.backward()

    should_update = (
        micro_step % accumulation_steps == 0
        or micro_step == len(train_loader)
    )
    if should_update:
        # Clip the complete accumulated gradient, not each partial gradient.
        clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
```

</details>

Accumulation reduces activation memory because only one micro-batch is resident at a time, but it does not reduce parameter or optimizer-state memory. It also does not provide the throughput of a physically larger batch: the micro-batches are still processed sequentially. In DDP, unnecessary gradient synchronization can be avoided for non-final micro-batches with the framework's `no_sync()` mechanism.

#### **Mixed Precision Training** {#mixed-precision-training}

Mixed precision executes suitable matrix operations in a lower-precision format while retaining higher precision where numerical range or accumulation accuracy matters. It can reduce activation memory, increase accelerator throughput, and make a larger micro-batch possible.

The common formats have different numerical properties:

| Format | Storage | Exponent range | Precision | Typical role |
|---|---:|---|---|---|
| FP32 | 32 bits | wide | high | reference training, sensitive reductions |
| FP16 | 16 bits | narrow | higher mantissa precision than BF16 | fast training with loss scaling |
| BF16 | 16 bits | similar exponent range to FP32 | lower mantissa precision | stable Transformer training on supported hardware |

FP16 gradients can underflow to zero because its exponent range is limited. Dynamic loss scaling multiplies the loss before backpropagation, checks for overflow, then unscales gradients before clipping and the optimizer step. BF16 usually does not require loss scaling because it preserves FP32-like exponent range, although sensitive operations may still run in FP32.

> ![Mixed precision training workflow](https://sebastianraschka.com/images/blog/2023/pytorch-memory-optimization/8_mixed-training.webp)
>
> Source: [Optimizing Memory Usage for Training LLMs and Vision Transformers in PyTorch](https://sebastianraschka.com/blog/2023/pytorch-memory-optimization.html)

<details>
<summary>Python Automatic Mixed Precision with FP16 or BF16</summary>

```python
import torch
from torch.nn.utils import clip_grad_norm_

device = "cuda"
use_bfloat16 = torch.cuda.is_bf16_supported()
amp_dtype = torch.bfloat16 if use_bfloat16 else torch.float16

# FP16 needs dynamic scaling more often. BF16 normally does not.
scaler = torch.amp.GradScaler(
    device,
    enabled=(amp_dtype == torch.float16),
)

for batch in train_loader:
    optimizer.zero_grad(set_to_none=True)

    # Autocast selects a safe dtype for each supported operation.
    with torch.autocast(device_type=device, dtype=amp_dtype):
        outputs = model(
            input_ids=batch["input_ids"].to(device),
            attention_mask=batch["attention_mask"].to(device),
            labels=batch["labels"].to(device),
        )
        loss = outputs.loss

    scaler.scale(loss).backward()

    # Gradients must be unscaled before measuring or clipping their norm.
    scaler.unscale_(optimizer)
    clip_grad_norm_(model.parameters(), max_norm=1.0)

    # For FP16, the step is skipped automatically if overflow is detected.
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()
```

</details>

Mixed precision should be validated rather than assumed correct. Compare the loss curve and final metrics with a short FP32 run, monitor skipped steps and `NaN` values, and keep numerically sensitive custom operations in FP32. Lower precision reduces the bytes per tensor, but it does not remove the need to understand optimizer and activation memory.

#### **Activation Checkpointing** {#activation-checkpointing}

Backpropagation normally needs intermediate activations from the forward pass. Activation checkpointing saves only selected boundary activations and discards the rest. During backward, the missing forward computation is repeated to reconstruct the values needed for gradients.

```text
ordinary training:
forward -> save every selected activation -> backward reuses them

activation checkpointing:
forward -> save checkpoint boundaries -> recompute segment -> backward
```

This exchanges compute for memory. It is most useful when activations dominate memory, especially with deep Transformers, long sequences, or large micro-batches. It does not shrink parameters or Adam moments, and it is unrelated to a training checkpoint saved to disk.

If a network is divided into $S$ segments, a suitable checkpoint schedule can reduce the number of simultaneously stored activations substantially, while adding an extra partial forward pass. The exact benefit depends on layer sizes and framework scheduling, so peak allocated memory and step time should both be measured.

<details>
<summary>Python Enabling Transformer Activation Checkpointing</summary>

```python
from torch.utils.checkpoint import checkpoint

# Hugging Face Transformer models expose the common implementation directly.
model.gradient_checkpointing_enable(
    gradient_checkpointing_kwargs={"use_reentrant": False}
)

# The same principle can be applied to an individual block.
def checkpointed_block(block, hidden_states, attention_mask):
    def run_block(states):
        return block(
            states,
            attention_mask=attention_mask,
        )[0]

    # The block's forward computation is repeated during backward.
    return checkpoint(run_block, hidden_states, use_reentrant=False)
```

</details>

Checkpointed regions must behave deterministically under recomputation. Random operations such as dropout require correct random-number-state handling, and in-place mutation can invalidate reconstruction. A practical configuration combines activation checkpointing with mixed precision, then chooses the largest micro-batch that still leaves safety margin.

#### **Distributed Data Parallel** {#distributed-data-parallel}

Distributed Data Parallel (DDP) replicates the model on each worker and gives each worker a different part of the mini-batch. Every worker computes local gradients, after which a collective operation aggregates them so that all replicas apply the same update.

For $D$ workers, the synchronized gradient is commonly the average:

$$
g=\frac{1}{D}\sum_{d=1}^{D}g^{(d)}
$$

The training flow is:

```text
shard input data across workers
-> run forward and backward independently
-> all-reduce gradient buckets
-> apply the same optimizer update on every replica
```

DDP improves throughput when the model fits on each device. It does not solve model-memory pressure because every worker still stores a complete model, gradients, and optimizer states.

<details>
<summary>Python Minimal PyTorch DDP Setup</summary>

```python
import os
import torch
import torch.distributed as dist
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.utils.data import DataLoader, DistributedSampler

def setup_ddp():
    # torchrun provides LOCAL_RANK, RANK, and WORLD_SIZE.
    local_rank = int(os.environ["LOCAL_RANK"])
    torch.cuda.set_device(local_rank)
    dist.init_process_group(backend="nccl")
    return local_rank

local_rank = setup_ddp()

model = build_model().to(local_rank)
model = DDP(model, device_ids=[local_rank])

sampler = DistributedSampler(
    train_dataset,
    shuffle=True,
    drop_last=False,
)
train_loader = DataLoader(
    train_dataset,
    batch_size=micro_batch_size,
    sampler=sampler,
)

for epoch in range(num_epochs):
    # Give each epoch a new, shared shuffling seed.
    sampler.set_epoch(epoch)

    for batch in train_loader:
        optimizer.zero_grad(set_to_none=True)
        loss = model(**move_to_device(batch, local_rank)).loss
        loss.backward()       # DDP synchronizes gradient buckets.
        optimizer.step()      # Every replica now applies the same update.

dist.destroy_process_group()
```

</details>

Correctness depends on more than launching several processes. Samples should not be duplicated accidentally, metrics must be aggregated across workers, random seeds should produce intended variation, and all workers must execute compatible collective operations. A hang often indicates that one worker skipped a collective because of divergent control flow or an earlier exception.

#### **Model, Tensor, and Pipeline Parallelism** {#model-tensor-and-pipeline-parallelism}

Data parallelism splits examples while replicating the model. Model parallelism is the broader family that splits the model itself. Two important forms are tensor parallelism and pipeline parallelism.

Tensor parallelism partitions the computation inside a layer. For example, different devices may own different column blocks of a projection matrix or different attention heads. Partial results are combined with operations such as all-reduce or all-gather. It is appropriate when a layer or its matrix operations cannot fit efficiently on one device, but communication occurs inside many layers.

Pipeline parallelism places consecutive layer groups on different devices. A batch is divided into micro-batches that move through the stages like an assembly line. It reduces per-device model depth, but the pipeline is not always full: startup and drain periods create a bubble.

| Strategy | Partitioned object | Main communication | Main limitation |
|---|---|---|---|
| data parallelism | input examples | gradient all-reduce | full model is replicated |
| tensor parallelism | tensors within a layer | all-reduce/all-gather inside layers | communication is frequent |
| pipeline parallelism | consecutive layer groups | activations and gradients between stages | idle pipeline bubbles |
| sequence/context parallelism | sequence dimension | distributed attention-related collectives | complex long-context kernels |

With $P$ pipeline stages and $M$ micro-batches, a simple fill-and-drain schedule has an approximate bubble fraction:

$$
\text{bubble fraction}\approx\frac{P-1}{M+P-1}
$$

More micro-batches reduce the bubble, but they increase scheduling complexity and may alter the effective batch. Real systems often combine dimensions:

```text
global cluster
-> data-parallel groups
-> tensor-parallel ranks inside each group
-> optional pipeline stages across layer ranges
```

The best layout depends on model shape, interconnect bandwidth, sequence length, and the failure domain. Parallelism that is efficient within a high-bandwidth node may be slow across nodes. Profiling communication overlap and achieved tokens per second is more informative than counting devices alone.

#### **ZeRO and Fully Sharded Data Parallel** {#zero-and-fully-sharded-data-parallel}

ZeRO reduces replicated training state across data-parallel workers. Instead of making every worker retain every optimizer state, gradient, and parameter at all times, progressively stronger stages partition those tensors across workers.

> ![ZeRO memory reduction stages](assets/zero-memory-stages.png)
>
> Source: [ZeRO: Memory Optimizations Toward Training Trillion Parameter Models](https://arxiv.org/abs/1910.02054)

| ZeRO idea | State sharded across workers | Main memory effect |
|---|---|---|
| stage 1 | optimizer states | removes duplicated Adam moments |
| stage 2 | optimizer states and gradients | also removes duplicated gradients |
| stage 3 | optimizer states, gradients, and parameters | full model need not remain resident on every worker |

Fully Sharded Data Parallel (FSDP) follows the stage-3 idea. Before a wrapped module computes, its parameter shards are gathered; after use, full parameters can be released. During backward, gradients are reduced and re-sharded. The conceptual cycle is:

```text
all-gather parameter shards for current module
-> compute forward or backward
-> reduce-scatter gradients
-> keep only the local shard
```

FSDP lowers persistent model-state memory, but it increases collective communication and makes wrapping policy, prefetching, mixed precision, and checkpoint format important. Extremely small wrapped units communicate too often; extremely large units create high peak memory during all-gather.

<details>
<summary>Python Minimal FSDP Wrapping Sketch</summary>

```python
import os
import torch
import torch.distributed as dist
from torch.distributed.fsdp import (
    FullyShardedDataParallel as FSDP,
    MixedPrecision,
)

dist.init_process_group(backend="nccl")
local_rank = int(os.environ["LOCAL_RANK"])
torch.cuda.set_device(local_rank)

mp_policy = MixedPrecision(
    param_dtype=torch.bfloat16,
    reduce_dtype=torch.float32,
    buffer_dtype=torch.bfloat16,
)

model = build_model().to(local_rank)

# Production code normally adds an auto-wrap policy so that Transformer
# blocks are gathered and released at useful boundaries.
model = FSDP(
    model,
    device_id=local_rank,
    mixed_precision=mp_policy,
    use_orig_params=True,
)

for batch in train_loader:
    optimizer.zero_grad(set_to_none=True)
    loss = model(**move_to_device(batch, local_rank)).loss
    loss.backward()
    optimizer.step()
```

</details>

Sharded checkpoints require an explicit save and restore policy. A local-shard checkpoint is efficient for resuming the same distributed layout, while a consolidated state dictionary is more portable but may require substantial CPU memory and communication.

The techniques solve different bottlenecks and are often composed:

| Situation | Reasonable starting point |
|---|---|
| model fits, batch does not | mixed precision + gradient accumulation |
| activations dominate | mixed precision + activation checkpointing |
| model fits on each GPU and throughput is low | DDP |
| full model states do not fit | FSDP/ZeRO |
| individual layers do not fit | tensor parallelism |
| many layers must be divided | pipeline parallelism, often with tensor/data parallelism |

Scaling should preserve a known-correct single-device baseline. Before trusting a larger run, compare one update with controlled seeds, verify the global batch and scheduler step count, aggregate metrics correctly, and report both model quality and systems measurements such as peak memory, tokens per second, and accelerator utilization.

### **Regularization and Generalization** {#regularization-and-generalization}

Generalization means that a model performs well on examples it did not see during training. In NLP, this is the real goal. A sentiment classifier should work on new reviews, a named entity recognizer should handle new documents, and a question-answering model should not collapse when the domain changes from Wikipedia to news or biomedical text.

Regularization is the collection of techniques used to improve generalization. It does not simply mean making the training loss smaller. In fact, some regularization methods deliberately make training harder so that the model cannot memorize the training set too easily.

A useful way to think about the problem is:

```text
training performance tells us how well the model fits seen data
validation performance tells us how well the model transfers to unseen data
regularization tries to reduce the gap between them
```

In neural NLP, overfitting often appears because pretrained models are large, task-specific datasets are small, labels may be noisy, and text often contains superficial shortcuts. A model may learn that a particular phrase, name, punctuation pattern, or dataset artifact predicts the label, even when that pattern is not truly meaningful.

#### **Overfitting and Underfitting** {#overfitting-and-underfitting}

Overfitting and underfitting describe two opposite failure modes.

Underfitting happens when the model is too weak, trained too little, or optimized poorly. It performs badly on both training data and validation data. The model has not captured the important patterns.

Overfitting happens when the model fits the training data too closely but fails to generalize. Training loss keeps decreasing, but validation loss stops improving or starts increasing. The model has learned details that are specific to the training set rather than patterns that transfer.

> Image for overfitting curves:
>
> ![Overfitting training and validation curves](https://upload.wikimedia.org/wikipedia/commons/thumb/1/1f/Overfitting_svg.svg/330px-Overfitting_svg.svg.png)
>
> Source: [Wikipedia - Overfitting](https://en.wikipedia.org/wiki/Overfitting)

The usual training curves look like this:

```text
underfitting:
training loss high, validation loss high

healthy fitting:
training loss decreases, validation loss decreases

overfitting:
training loss keeps decreasing, validation loss increases
```

| Situation | Training Loss | Validation Loss | Interpretation |
|---|---|---|---|
| underfitting | high | high | model has not learned enough |
| good fit | low | low | model learned transferable patterns |
| overfitting | very low | high or rising | model memorized training-specific patterns |

In NLP, overfitting can be subtle. A model may achieve high validation accuracy on a random split but fail on a different domain. For example, a fake-news classifier trained on one dataset may learn publisher-specific wording rather than general evidence of misinformation. A review classifier may associate long reviews with positive sentiment because of dataset bias.

A quick diagnostic pattern is:

```text
train score much higher than validation score -> likely overfitting
both scores low -> likely underfitting or optimization problem
validation score unstable -> dataset may be small or learning rate too high
```

<details>
<summary>Python Detecting Overfitting from Training History</summary>

```python
train_loss = [0.95, 0.70, 0.48, 0.30, 0.18, 0.10]
valid_loss = [1.02, 0.82, 0.65, 0.63, 0.71, 0.88]

best_epoch = min(range(len(valid_loss)), key=lambda i: valid_loss[i])

print("best validation epoch:", best_epoch + 1)
print("best validation loss:", valid_loss[best_epoch])

if train_loss[-1] < train_loss[best_epoch] and valid_loss[-1] > valid_loss[best_epoch]:
    print("Warning: training loss improved after the best validation epoch, but validation loss got worse.")
    print("This is a typical overfitting pattern.")
```

</details>

Regularization methods such as dropout, weight decay, early stopping, and data augmentation are different ways to reduce this gap.

#### **Dropout** {#dropout}

Dropout randomly disables some hidden units during training. At each training step, the model samples a different subnetwork. This prevents the model from relying too heavily on any single neuron or feature path.

> Image for dropout:
>
> ![Dropout neural network diagram](https://cdn-ak.f.st-hatena.com/images/fotolife/s/sonickun/20160714/20160714170858.jpg)
>
> Source: [sonickun.log - Overfitting and Dropout](https://sonickun.hatenablog.com/entry/2016/07/18/191656)

If $h$ is a hidden representation, dropout can be written as:

$$
\tilde{h} = \frac{m \odot h}{1 - p}, \qquad m_i \sim \text{Bernoulli}(1 - p)
$$

Here, $p$ is the dropout probability. If $p = 0.1$, about 10% of units are dropped. The mask $m$ decides which units are kept. The symbol $\odot$ means element-wise multiplication. The division by $1-p$ keeps the expected scale of activations roughly stable during training.

The important detail is that dropout is active only during training. During evaluation and inference, dropout is turned off.

```text
training mode:    random units removed
inference mode:   full network used
```

In NLP, dropout is commonly applied to embeddings, hidden states, attention outputs, feed-forward layers, and classification heads. Transformer models often use multiple dropout locations because overfitting can happen at different parts of the network.

<details>
<summary>Python Dropout in a Text Classifier</summary>

```python
import torch
import torch.nn as nn

class DropoutTextClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes, dropout_p=0.2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.encoder = nn.Linear(embed_dim, hidden_dim)
        self.dropout = nn.Dropout(p=dropout_p)
        self.classifier = nn.Linear(hidden_dim, num_classes)

    def forward(self, input_ids):
        # Step 1: convert token IDs to embeddings.
        embeddings = self.embedding(input_ids)

        # Step 2: mean-pool tokens into one sentence representation.
        sentence_vector = embeddings.mean(dim=1)

        # Step 3: build a hidden representation.
        hidden = torch.relu(self.encoder(sentence_vector))

        # Step 4: randomly drop hidden units during training.
        hidden = self.dropout(hidden)

        # Step 5: classify using the regularized representation.
        logits = self.classifier(hidden)
        return logits

model = DropoutTextClassifier(
    vocab_size=10000,
    embed_dim=128,
    hidden_dim=256,
    num_classes=2,
    dropout_p=0.2
)

model.train()  # dropout is active
train_logits = model(torch.randint(0, 10000, (4, 20)))

model.eval()   # dropout is disabled
valid_logits = model(torch.randint(0, 10000, (4, 20)))
```

</details>

Dropout is useful, but too much dropout can cause underfitting. A small text classifier may not need heavy dropout. A large pretrained model fine-tuned on a small dataset may benefit from modest dropout in the classification head.

#### **Weight Decay** {#weight-decay}

Weight decay discourages parameters from becoming too large. Large weights can make a model overly sensitive to small changes in input. In NLP, this can mean the model reacts too strongly to a few words, punctuation marks, or dataset-specific artifacts.

A common regularized objective is:

$$
L_{\text{total}} = L_{\text{task}} + \lambda \lVert \theta \rVert_2^2
$$

Here, $L_{\text{task}}$ is the original loss, such as cross-entropy. The second term penalizes large parameter values. The coefficient $\lambda$ controls how strong the penalty is.

The intuition is:

```text
small weights -> smoother model -> less sensitivity -> better generalization
```

Weight decay is closely related to L2 regularization, but in adaptive optimizers such as Adam, the implementation detail matters. AdamW is commonly preferred because it decouples weight decay from Adam's adaptive gradient update. This makes the regularization behavior cleaner and more predictable.

In Transformer fine-tuning, a common practice is to apply weight decay to most weight matrices but not to bias terms and normalization parameters. Biases and LayerNorm weights usually do not benefit from the same decay.

<details>
<summary>Python AdamW with Selective Weight Decay</summary>

```python
from torch.optim import AdamW

no_decay = ["bias", "LayerNorm.weight", "layer_norm.weight"]

optimizer_grouped_parameters = [
    {
        "params": [
            param for name, param in model.named_parameters()
            if not any(key in name for key in no_decay)
        ],
        "weight_decay": 0.01,
    },
    {
        "params": [
            param for name, param in model.named_parameters()
            if any(key in name for key in no_decay)
        ],
        "weight_decay": 0.0,
    },
]

optimizer = AdamW(
    optimizer_grouped_parameters,
    lr=2e-5
)
```

</details>

Weight decay is not a replacement for validation monitoring. If it is too small, it may do almost nothing. If it is too large, it can prevent the model from adapting to the task.

#### **Early Stopping** {#early-stopping}

Early stopping stops training when validation performance stops improving. It is a simple but powerful form of regularization because it prevents the model from continuing to specialize on the training set after validation quality has peaked.

The logic is:

```text
keep training while validation improves
save the best checkpoint
stop after patience runs out
restore the best checkpoint
```

The key idea is that training time itself controls model complexity. A model trained for too few epochs may underfit. A model trained for too many epochs may overfit. Early stopping chooses the stopping point based on validation behavior.

Important terms:

| Term | Meaning |
|---|---|
| monitor metric | validation loss, accuracy, F1, BLEU, etc. |
| patience | how many evaluations to wait without improvement |
| best checkpoint | model state with the best validation metric |
| restore best weights | use the best model, not the final model |

For NLP classification, validation loss is often more sensitive than accuracy. For sequence generation, metrics such as BLEU, ROUGE, or validation perplexity may be monitored depending on the task.

<details>
<summary>Python Early Stopping Sketch</summary>

```python
import copy

best_valid_loss = float("inf")
best_state_dict = None
patience = 3
num_bad_epochs = 0

for epoch in range(num_epochs):
    train_one_epoch(model, train_loader, optimizer, scheduler)
    valid_loss = evaluate_loss(model, valid_loader)

    print(f"epoch={epoch + 1}, valid_loss={valid_loss:.4f}")

    if valid_loss < best_valid_loss:
        # Validation improved, so keep this checkpoint.
        best_valid_loss = valid_loss
        best_state_dict = copy.deepcopy(model.state_dict())
        num_bad_epochs = 0
    else:
        # Validation did not improve.
        num_bad_epochs += 1

    if num_bad_epochs >= patience:
        print("Early stopping triggered.")
        break

# Restore the best validation checkpoint before final testing.
model.load_state_dict(best_state_dict)
```

</details>

Early stopping is especially useful when datasets are small, when training is expensive, or when the correct number of epochs is unclear. It should be based on a validation set that represents the real target distribution as much as possible.

#### **Data Augmentation in NLP** {#data-augmentation-in-nlp}

Data augmentation creates additional training examples from existing data. The goal is to expose the model to more linguistic variation so that it learns the underlying task rather than memorizing surface forms.

In computer vision, augmentation is often intuitive: rotate, crop, resize, or change brightness. In NLP, augmentation is harder because small text changes can change meaning. For example, changing "good" to "bad" flips sentiment. Removing "not" from "not useful" changes the label completely.

Useful NLP augmentation must preserve the label or deliberately update the label.

Common methods include:

| Method | Example | Risk |
|---|---|---|
| synonym replacement | "great movie" -> "excellent movie" | synonym may not fit context |
| random deletion | remove unimportant words | may remove meaning-bearing words |
| word order perturbation | slightly reorder local phrases | can break grammar |
| back-translation | English -> French -> English | translation may change nuance |
| prompt-based generation | ask an LLM to create more examples | may introduce label noise |
| noise injection | typos, casing, punctuation changes | may make text unrealistic |

> Image for NLP data augmentation:
>
> ![Data augmentation techniques for robust question answering](https://web.stanford.edu/class/archive/cs/cs224n/cs224n.1214/reports/final_summaries/images/image141.png)
>
> Source: [Stanford CS224N Final Project Summaries](https://web.stanford.edu/class/archive/cs/cs224n/cs224n.1214/reports/final_summaries/default.html)

Back-translation is a classic example. A sentence is translated into another language and then translated back. The result often has the same meaning but different wording.

```text
original:       The movie was surprisingly good.
translation:    Le film était étonnamment bon.
back-translated:The film was unexpectedly good.
```

This can help the model become less dependent on one exact phrasing.

<details>
<summary>Python Simple Text Augmentation Examples</summary>

```python
import random

synonyms = {
    "good": ["great", "excellent", "positive"],
    "bad": ["poor", "terrible", "negative"],
    "movie": ["film"],
    "fast": ["quick", "rapid"]
}

def synonym_replace(text, replace_prob=0.3):
    """Replace some words with simple predefined synonyms."""
    new_words = []

    for word in text.split():
        key = word.lower().strip(".,!?;:")

        if key in synonyms and random.random() < replace_prob:
            new_words.append(random.choice(synonyms[key]))
        else:
            new_words.append(word)

    return " ".join(new_words)


def random_deletion(text, delete_prob=0.1):
    """Randomly remove a small number of words."""
    words = text.split()

    # Keep at least one word so the example does not become empty.
    kept_words = [word for word in words if random.random() > delete_prob]
    if not kept_words:
        kept_words = [random.choice(words)]

    return " ".join(kept_words)

sentence = "The movie was good and the story was fast"

print("original: ", sentence)
print("synonym:  ", synonym_replace(sentence))
print("deleted:  ", random_deletion(sentence))
```

</details>

For high-stakes tasks, augmentation should be checked carefully. If augmented examples change the label, the model may learn contradictions. In sentiment analysis, replacing words is risky. In named entity recognition, replacing a name may require updating entity labels. In question answering, changing a question or context may invalidate the answer span.

A practical summary:

| Technique | What It Controls | Good For | Main Risk |
|---|---|---|---|
| dropout | hidden feature reliance | neural models with many parameters | too much dropout causes underfitting |
| weight decay | parameter magnitude | Transformer fine-tuning, smooth models | too much decay blocks adaptation |
| early stopping | training duration | small datasets, expensive training | bad validation set gives bad stopping point |
| data augmentation | training data variety | low-resource and domain shift settings | label noise or meaning drift |

Regularization should be chosen based on the failure pattern. If the model memorizes quickly, use dropout, weight decay, and early stopping. If the model fails on new wording or new domains, data augmentation may help. If both training and validation are poor, the problem is probably not regularization; it may require a better model, better optimization, cleaner labels, or more informative features.

### **Pretraining, Transfer Learning, and Task Adaptation** {#pretraining-transfer-learning-and-task-adaptation}

Transfer learning separates the acquisition of broad language regularities from adaptation to a particular domain, task, or interaction format. A model may first learn from a large general corpus, continue self-supervised learning on target-domain text, receive supervised task demonstrations, and finally be adapted with either all parameters or a small trainable subset.

These stages answer different questions:

| Stage | Main data | Objective | Main purpose |
|---|---|---|---|
| general pretraining | broad unlabelled corpus | MLM, CLM, or denoising | acquire general representations and generation ability |
| domain/task-adaptive pretraining | unlabelled target-like text | original self-supervised objective | reduce domain and vocabulary mismatch |
| supervised fine-tuning | labelled or input-output examples | task likelihood | teach a target mapping or response format |
| instruction tuning | diverse instruction-response data | response likelihood | generalize across natural-language task descriptions |
| PEFT or full fine-tuning | any adaptation dataset | depends on the stage | choose which parameters are allowed to change |

The final row is deliberately different from the others. Full fine-tuning and PEFT are parameter-update strategies, not data stages. Instruction tuning can be performed with full fine-tuning, LoRA, QLoRA, or another PEFT method.

#### **Why Pretraining Matters** {#why-pretraining-matters}

Pretraining matters because labelled NLP data is usually expensive, limited, and domain-specific, while unlabelled text is abundant. A model can learn a lot about language without manual labels by solving self-supervised tasks.

Common pretraining objectives include:

| Model Style | Objective | Example |
|---|---|---|
| encoder-only | masked language modeling | predict `[MASK]` from both left and right context |
| decoder-only | causal language modeling | predict the next token from previous tokens |
| encoder-decoder | denoising or seq2seq reconstruction | recover corrupted text or map input to target text |

For a causal language model, the objective is usually next-token prediction:

$$
L = -\sum_{t=1}^{T} \log p(x_t \mid x_{<t})
$$

Here, $x_t$ is the current token, and $x_{<t}$ means all previous tokens. The model is rewarded when it assigns high probability to the correct next token.

For masked language modeling, the loss is computed only on selected masked positions:

$$
L = -\sum_{i \in M} \log p(x_i \mid x_{\setminus M})
$$

Here, $M$ is the set of masked token positions. The model sees the surrounding context and predicts the original hidden tokens.

The important idea is that pretraining creates a strong initialization. The model does not start fine-tuning from random weights. It starts from a region of parameter space that already encodes useful language behavior.

```text
without pretraining: random weights -> learn language + task
with pretraining:    language-aware weights -> learn task adaptation
```

<details>
<summary>Python Sketch: Loading a Pretrained Model</summary>

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification

model_name = "bert-base-uncased"

# Step 1: load the tokenizer learned during pretraining.
tokenizer = AutoTokenizer.from_pretrained(model_name)

# Step 2: load the pretrained encoder and attach a new classification head.
# The backbone starts from pretrained weights.
# The classification head is usually newly initialized.
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

text = "This course is surprisingly useful."
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True)
outputs = model(**inputs)

print(outputs.logits)
```

</details>

Pretraining is not magic. If the downstream domain is very different, such as clinical notes, legal contracts, or code, general pretraining may not be enough. In that case, domain-adaptive pretraining or careful fine-tuning may be needed.

#### **Feature Extraction, Full Fine-Tuning, and PEFT** {#feature-extraction-full-fine-tuning-and-peft}

Feature extraction and fine-tuning are two ways to use a pretrained model.

In feature extraction, the pretrained backbone is frozen. It converts text into representations, and only a small task-specific model is trained on top. The pretrained model acts like a feature generator.

In fine-tuning, some or all pretrained parameters are updated. The model representation itself changes to fit the downstream task.

```text
feature extraction:
text -> frozen pretrained model -> fixed representation -> train small head

fine-tuning:
text -> pretrained model -> update backbone and head together
```

Feature extraction is cheaper and safer. It is useful when the dataset is small, compute is limited, or we want to avoid disturbing the pretrained model. Full fine-tuning is more flexible and often performs better, but it has higher memory cost and a greater risk of overfitting or catastrophic forgetting.

<details>
<summary>Python Feature Extraction with Frozen Backbone</summary>

```python
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer

class FrozenBertClassifier(nn.Module):
    def __init__(self, model_name="bert-base-uncased", num_labels=2):
        super().__init__()
        self.backbone = AutoModel.from_pretrained(model_name)
        self.classifier = nn.Linear(self.backbone.config.hidden_size, num_labels)

        # Freeze the pretrained backbone.
        for param in self.backbone.parameters():
            param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        # Step 1: get contextual representations from the frozen model.
        with torch.no_grad():
            outputs = self.backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Step 2: use the [CLS] representation for classification.
        cls_vector = outputs.last_hidden_state[:, 0, :]

        # Step 3: train only the classifier head.
        logits = self.classifier(cls_vector)
        return logits
```

</details>

| Method | What Is Updated? | Strength | Weakness | Good For |
|---|---|---|---|---|
| feature extraction | task head only | cheap, stable, low overfitting risk | less adaptable | small datasets, quick baselines |
| partial fine-tuning | selected layers and task head | balance of efficiency and flexibility | layer choice matters | limited compute |
| full fine-tuning | all parameters | strongest adaptation | expensive, can forget | high-value downstream tasks |
| PEFT | small adapter/prompt/LoRA parameters | efficient and reusable | extra method complexity | large models and many tasks |

Full fine-tuning updates all model parameters on the downstream dataset. The pretrained model is used as initialization, but every layer is allowed to move.

This is often the strongest method when enough labelled data and compute are available. It lets early layers, middle layers, attention patterns, and output heads all adjust to the task. For example, a biomedical NER model may need token representations that are sensitive to medical terminology. A legal document classifier may need attention patterns that track long clause structures.

The risk is that full fine-tuning can overwrite useful pretrained knowledge. This is why fine-tuning typically uses small learning rates, short training, warmup, weight decay, and validation monitoring.

A common fine-tuning loss for classification is:

$$
L = -\sum_{c=1}^{C} y_c \log \hat{y}_c
$$

Here, $C$ is the number of classes, $y_c$ is the gold one-hot label, and $\hat{y}_c$ is the predicted class probability. The loss updates both the classification head and the pretrained backbone.

<details>
<summary>Python Full Fine-Tuning Sketch</summary>

```python
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.optim import AdamW

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2
)

# In full fine-tuning, all parameters remain trainable.
for name, param in model.named_parameters():
    param.requires_grad = True

optimizer = AdamW(model.parameters(), lr=2e-5, weight_decay=0.01)

model.train()
for batch in dataloader:
    optimizer.zero_grad()

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()
    scheduler.step()
```

</details>

Full fine-tuning is usually a good baseline for medium-sized pretrained models. For very large models, however, it can become expensive because optimizer states, gradients, and checkpoint copies must be stored for all parameters.

#### **Domain-Adaptive and Task-Adaptive Pretraining** {#domain-and-task-adaptive-pretraining}

A pretrained model can understand general language yet remain poorly matched to clinical notes, legal contracts, scientific papers, source code, historical text, or a specialized organization. Domain-adaptive pretraining (DAPT) continues the original self-supervised objective on a sizeable target-domain corpus. Task-adaptive pretraining (TAPT) uses the smaller unlabelled text associated with the downstream task, often including the task's inputs without their labels.

```text
general pretrained model
-> DAPT on broad target-domain text
-> TAPT on task-specific unlabelled inputs
-> supervised task adaptation
```

The method does not require new manual labels. For an encoder, one may continue masked language modeling; for a decoder-only model, one may continue causal language modeling. The benefit comes from adjusting representations toward the target terminology, style, entity distribution, and discourse patterns before the scarce supervised signal is used.

The same self-supervised loss remains:

$$
\mathcal{L}_{\text{adapt}}
=-\sum_{t\in\mathcal{T}}\log p_\theta(x_t\mid c_t)
$$

Here, $\mathcal{T}$ denotes the prediction positions and $c_t$ is the context permitted by the original objective. For a causal model, $c_t=x_{<t}$; for masked language modeling, it contains the visible left and right context.

<details>
<summary>Python Continued Masked-Language-Model Training Sketch</summary>

```python
from transformers import (
    AutoModelForMaskedLM,
    AutoTokenizer,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
)

model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForMaskedLM.from_pretrained(model_name)

def tokenize_domain_batch(batch):
    # Keep target-domain wording unchanged and create fixed-length chunks.
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=256,
    )

tokenized_domain_data = domain_dataset.map(
    tokenize_domain_batch,
    batched=True,
    remove_columns=domain_dataset.column_names,
)

# The collator selects masked positions dynamically on each epoch.
collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=True,
    mlm_probability=0.15,
)

trainer = Trainer(
    model=model,
    args=TrainingArguments(
        output_dir="domain-adapted-bert",
        learning_rate=5e-5,
        num_train_epochs=2,
        per_device_train_batch_size=16,
    ),
    train_dataset=tokenized_domain_data,
    data_collator=collator,
)
trainer.train()
```

</details>

Continued pretraining can also hurt. Too much narrow-domain training can cause catastrophic forgetting, duplicate benchmark text can create leakage, and low-quality web data can reinforce noise. The correct comparison includes the untouched pretrained model, direct supervised fine-tuning, DAPT/TAPT followed by fine-tuning, and evaluation on both target-domain and retained general-domain slices.

| Method | Unlabelled data required | Main strength | Main risk |
|---|---:|---|---|
| direct fine-tuning | no | simplest and cheapest | domain mismatch remains |
| TAPT | small task corpus | closely matches task inputs | easy to overfit a narrow distribution |
| DAPT | larger domain corpus | learns broad domain language | more compute and possible forgetting |
| DAPT + TAPT | both | staged specialization | more hyperparameters and provenance checks |

#### **Supervised Fine-Tuning** {#supervised-fine-tuning}

Supervised fine-tuning (SFT) trains a pretrained model on examples with desired outputs. For a classifier, the desired output is a label. For a sequence model, it may be a translation, summary, structured record, tool call, or assistant response. SFT teaches the conditional mapping that pretraining alone does not specify.

For an autoregressive response $y=(y_1,\ldots,y_T)$ conditioned on prompt $x$, the common objective is:

$$
\mathcal{L}_{\text{SFT}}
=-\sum_{t=1}^{T}\log \pi_\theta(y_t\mid x,y_{<t})
$$

The model normally receives both prompt and response tokens, but prompt tokens can be assigned the ignore index so that only the desired response contributes to loss. This distinction matters: training on arbitrary prompt text as a target can encourage prompt repetition rather than answer generation.

<details>
<summary>Python Response-Only Label Construction</summary>

```python
IGNORE_INDEX = -100

def build_sft_example(tokenizer, prompt, response, max_length=512):
    """Tokenize prompt and response while training only on the response."""
    prompt_ids = tokenizer(
        prompt,
        add_special_tokens=False,
    )["input_ids"]
    response_ids = tokenizer(
        response + tokenizer.eos_token,
        add_special_tokens=False,
    )["input_ids"]

    input_ids = (prompt_ids + response_ids)[:max_length]

    # Prompt positions are visible to the model but ignored by the loss.
    labels = (
        [IGNORE_INDEX] * len(prompt_ids) + response_ids
    )[:max_length]

    attention_mask = [1] * len(input_ids)
    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "labels": labels,
    }

example = build_sft_example(
    tokenizer,
    prompt="Summarize: Gradient clipping limits unstable updates.\nAnswer:",
    response="Gradient clipping rescales excessively large gradients.",
)
```

</details>

SFT quality depends strongly on demonstration quality, coverage, and formatting. A small collection of consistent expert outputs may be more valuable than a large mixture containing contradictions, copied boilerplate, or unsupported answers. Exact prompt templates, role separators, end-of-sequence handling, truncation policy, and loss masking should be versioned with the dataset.

SFT optimizes imitation, not preference among all valid responses. If demonstrations are concise, the model learns concise answers; if they contain unsupported certainty, that behavior is also imitated. Preference optimization may refine choices after SFT, but it cannot recover information or capabilities absent from the model and data.

#### **Instruction Tuning** {#instruction-tuning}

Instruction tuning is supervised fine-tuning on examples written as natural-language instructions and desired outputs. It teaches a language model to follow user requests rather than only continue text in the style of pretraining data.

A typical instruction-tuning example looks like this:

```text
Instruction: Summarize the following paragraph in one sentence.
Input:       Neural NLP models can be expensive to train...
Output:      Neural NLP training requires careful optimization and memory management.
```

The model is trained to generate the output given the instruction and optional input. For decoder-only models, this is still next-token prediction, but the loss is usually applied to the response part rather than the prompt part.

> Image for instruction tuning:
>
> ![General pipeline of instruction tuning](https://ar5iv.labs.arxiv.org/html/2308.10792/assets/x1.png)
>
> Source: [Instruction Tuning for Large Language Models: A Survey](https://arxiv.org/abs/2308.10792)

Instruction tuning bridges a mismatch. During pretraining, a language model learns to predict the next token from web text, books, code, or other corpora. Users, however, usually want the model to answer, explain, translate, summarize, classify, or reason according to an instruction.

```text
pretraining objective:       continue text
instruction tuning objective: follow task description and produce target answer
```

<details>
<summary>Python Instruction Formatting and Label Masking</summary>

```python
IGNORE_INDEX = -100

def format_instruction_example(example):
    """Convert a structured example into an instruction-tuning text format."""
    instruction = example["instruction"].strip()
    input_text = example.get("input", "").strip()
    output = example["output"].strip()

    if input_text:
        prompt = f"### Instruction:\n{instruction}\n\n### Input:\n{input_text}\n\n### Response:\n"
    else:
        prompt = f"### Instruction:\n{instruction}\n\n### Response:\n"

    full_text = prompt + output
    return prompt, full_text

example = {
    "instruction": "Classify the sentiment as positive or negative.",
    "input": "The explanation was clear and useful.",
    "output": "positive"
}

prompt, full_text = format_instruction_example(example)

prompt_ids = tokenizer(prompt, add_special_tokens=False)["input_ids"]
full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"]

# The model sees the whole sequence, but we only train it on the response tokens.
labels = full_ids.copy()
labels[:len(prompt_ids)] = [IGNORE_INDEX] * len(prompt_ids)

print(tokenizer.decode(full_ids))
print(labels)
```

</details>

Instruction tuning improves usability, but it does not guarantee truthfulness, safety, or deep reasoning. It can teach style and format very effectively, but the quality depends strongly on instruction diversity, output quality, domain coverage, and evaluation design.

| Stage | Data | Training Signal | Main Effect |
|---|---|---|---|
| pretraining | large unlabelled text | self-supervised prediction | general language ability |
| full fine-tuning | labelled task data | supervised task loss | strong task adaptation |
| PEFT | labelled or instruction data | supervised loss on small trainable modules | efficient specialization |
| instruction tuning | instruction-output pairs | supervised response generation | instruction-following behavior |

A practical strategy is to start with feature extraction or full fine-tuning for small classification tasks, use AdamW with small learning rates for Transformer fine-tuning, and move to PEFT when the base model is too large or many task-specific variants are needed. For generative assistants, instruction tuning is usually the first supervised post-training step after pretraining.

### **Parameter-Efficient Fine-Tuning** {#parameter-efficient-fine-tuning}

Parameter-Efficient Fine-Tuning (PEFT) adapts a pretrained model while training only a small set of new or selected parameters. The frozen base model supplies most of the computation and knowledge; the trainable component changes how that knowledge is used for a task or domain.

PEFT is orthogonal to the training stage. LoRA can be used for classification fine-tuning, instruction tuning, continued adaptation, or preference optimization. The phrase describes **which parameters change**, not **which supervision signal is used**.

#### **Why Parameter Efficiency Matters** {#why-parameter-efficiency-matters}

Full fine-tuning stores gradients and optimizer states for every trainable parameter and usually produces a complete model checkpoint per task. With many large models or many customer/domain variants, this cost dominates experimentation and deployment storage.

If a base model has $N$ parameters and the adaptation has $n$ trainable parameters, the trainable fraction is:

$$
\rho=\frac{n}{N}\times 100\%
$$

A small $\rho$ reduces gradient and optimizer-state memory and makes task-specific checkpoints compact. It does **not** mean that inference loads only $n$ parameters: the base model is still required unless weights are distilled into another model.

| Property | Full fine-tuning | PEFT |
|---|---|---|
| trainable parameters | nearly all base parameters | a small subset or added modules |
| optimizer-state memory | high | much lower |
| task checkpoint size | roughly a full model | adapter-sized |
| adaptation capacity | maximum | constrained by method and rank |
| base-model sharing | difficult across many tasks | natural |
| inference speed | normal base model | often similar; adapters may add small overhead |

PEFT is particularly useful when compute is constrained, the dataset is small, one base model serves many tasks, or catastrophic forgetting must be limited. Full fine-tuning remains valuable when maximum adaptation is required and sufficient data, memory, and validation are available.

#### **Adapter-Based Fine-Tuning** {#adapter-based-fine-tuning}

An adapter inserts a small bottleneck network into each selected Transformer block while freezing the original block. A common residual adapter computes:

$$
h'=h+W_{\text{up}}\,\sigma(W_{\text{down}}h)
$$

$W_{\text{down}}$ projects hidden state $h\in\mathbb{R}^{d}$ into a smaller dimension $r$, the nonlinearity transforms it, and $W_{\text{up}}$ projects it back to $d$. When $r\ll d$, the trainable matrices are much smaller than the surrounding Transformer weights. The residual path preserves the original representation when the adapter update is small.

<details>
<summary>Python Bottleneck Adapter Module</summary>

```python
import torch.nn as nn

class BottleneckAdapter(nn.Module):
    def __init__(self, hidden_size, bottleneck_size=64, dropout=0.1):
        super().__init__()
        self.down = nn.Linear(hidden_size, bottleneck_size)
        self.activation = nn.GELU()
        self.dropout = nn.Dropout(dropout)
        self.up = nn.Linear(bottleneck_size, hidden_size)

        # Begin close to the frozen base model's behavior.
        nn.init.zeros_(self.up.weight)
        nn.init.zeros_(self.up.bias)

    def forward(self, hidden_states):
        update = self.down(hidden_states)
        update = self.activation(update)
        update = self.dropout(update)
        update = self.up(update)

        # The adapter learns a task-specific residual correction.
        return hidden_states + update
```

</details>

Adapters are modular and can be swapped without changing the base checkpoint. Their main cost is architectural: inserted modules add a sequential operation to the forward pass and may complicate serving systems that were optimized for the original model graph.

#### **LoRA** {#lora}

Low-Rank Adaptation (LoRA) assumes that the useful change to a pretrained weight matrix can be represented in a low-rank subspace. It freezes the original matrix $W_0\in\mathbb{R}^{d\times k}$ and learns two smaller matrices:

$$
W'x=W_0x+\frac{\alpha}{r}BAx
$$

where $A\in\mathbb{R}^{r\times k}$, $B\in\mathbb{R}^{d\times r}$, and rank $r$ is much smaller than $d$ and $k$. Instead of training $dk$ values for the full update, LoRA trains $r(k+d)$ values. The factor $\alpha/r$ controls update scale independently of rank.

> ![LoRA low-rank adaptation](assets/lora-low-rank-adaptation.png)
>
> Source: [LoRA: Low-Rank Adaptation of Large Language Models](https://arxiv.org/abs/2106.09685)

One factor is commonly initialized randomly and the other to zero, so the initial LoRA branch contributes no change. During training, gradients flow only through $A$ and $B$. At deployment, the update can often be merged into $W_0$, eliminating an additional matrix branch when only one adapter is needed.

<details>
<summary>Python LoRA Fine-Tuning with PEFT</summary>

```python
from peft import LoraConfig, TaskType, get_peft_model
from transformers import AutoModelForCausalLM

base_model = AutoModelForCausalLM.from_pretrained("gpt2")

lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,                    # rank of the trainable update
    lora_alpha=16,          # update scaling before division by rank
    lora_dropout=0.05,
    target_modules=["c_attn", "c_proj"],
    bias="none",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()

for batch in train_loader:
    optimizer.zero_grad(set_to_none=True)
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"],
    )
    outputs.loss.backward()
    optimizer.step()
```

</details>

Rank, target modules, scaling, and learning rate are genuine hyperparameters. Attention projections are common targets, but adapting feed-forward projections can help tasks requiring larger representation changes. A rank that is too small underfits; a very large rank erodes the memory and storage advantage.

#### **QLoRA** {#qlora}

QLoRA combines a frozen quantized base model with trainable LoRA adapters. The base weights are stored in 4-bit form for memory efficiency, dequantized to a computation dtype such as BF16 when used, and kept frozen. Gradients update the LoRA parameters, not the quantized base values.

> ![QLoRA memory comparison](assets/qlora-memory-comparison.png)
>
> Source: [QLoRA: Efficient Finetuning of Quantized LLMs](https://arxiv.org/abs/2305.14314)

The original QLoRA method highlights three systems ideas:

| Technique | Purpose |
|---|---|
| 4-bit NormalFloat (NF4) | represent normally distributed pretrained weights efficiently |
| double quantization | quantize quantization constants to reduce metadata memory |
| paged optimizers | reduce memory spikes by moving optimizer pages when necessary |

The computation can be summarized as:

```text
4-bit frozen base weight
-> dequantize for matrix computation
-> add BF16/FP16 LoRA update
-> backpropagate only into LoRA parameters
```

<details>
<summary>Python QLoRA Configuration Sketch</summary>

```python
import torch
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
from transformers import AutoModelForCausalLM, BitsAndBytesConfig

quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=quantization_config,
    device_map="auto",
)

# Prepare normalization and input-gradient handling for k-bit training.
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()
```

</details>

QLoRA can make single-device adaptation feasible, but it is not identical to ordinary LoRA. Quantization introduces approximation, supported kernels and hardware matter, and a 4-bit training checkpoint still requires the exact compatible base model and quantization configuration for restoration.

#### **Prompt Tuning and Prefix Tuning** {#prompt-and-prefix-tuning}

Prompt tuning learns a small sequence of continuous embeddings that is prepended to the input embedding sequence. The base model remains frozen, and the learned "soft prompt" steers its behavior without corresponding to readable words.

Prefix tuning learns continuous vectors that act as prefixes to attention key/value states, commonly at multiple Transformer layers. It influences attention throughout the network more directly than input-only prompt tuning.

| Method | Trainable object | Where it enters | Typical implication |
|---|---|---|---|
| prompt tuning | soft input embeddings | input embedding sequence | extremely compact, may need large models |
| prefix tuning | per-layer key/value-like prefixes | attention in multiple layers | more expressive, consumes context-related state |
| adapters | bottleneck modules | Transformer blocks | modular but adds forward operations |
| LoRA | low-rank weight updates | selected linear projections | strong default and mergeable |

<details>
<summary>Python Conceptual Soft-Prompt Module</summary>

```python
import torch
import torch.nn as nn

class SoftPrompt(nn.Module):
    def __init__(self, prompt_length, hidden_size):
        super().__init__()
        self.prompt = nn.Parameter(
            torch.empty(prompt_length, hidden_size)
        )
        nn.init.normal_(self.prompt, mean=0.0, std=0.02)

    def forward(self, token_embeddings):
        batch_size = token_embeddings.size(0)

        # Repeat the same learned prompt for every example in the batch.
        prompt = self.prompt.unsqueeze(0).expand(batch_size, -1, -1)
        return torch.cat([prompt, token_embeddings], dim=1)
```

</details>

Because the learned vectors are continuous, they are not directly interpretable as a natural-language prompt. Attention masks, position IDs, and maximum context length must be adjusted for the added positions.

#### **Choosing a PEFT Strategy** {#choosing-a-peft-strategy}

There is no universally best PEFT method. The decision depends on whether the main constraint is training memory, task checkpoint storage, serving latency, adapter swapping, or adaptation capacity.

| Requirement | Strong candidate | Reason |
|---|---|---|
| robust general default | LoRA | good quality-efficiency balance and broad tooling |
| base model barely fits | QLoRA | 4-bit base storage plus LoRA training |
| many modular task components | adapters | explicit interchangeable modules |
| smallest task state | prompt tuning | very few learned vectors |
| stronger attention steering without weight updates | prefix tuning | injects learned state across layers |
| maximum adaptation with sufficient resources | full fine-tuning | no low-rank or module constraint |

A fair comparison holds the dataset, objective, number of optimizer updates, validation protocol, and decoding settings constant. Report trainable parameters, peak memory, wall-clock time, checkpoint size, final task quality, and retained general capability. Parameter count alone can hide slower kernels or weaker adaptation.

PEFT also creates lifecycle questions. The adapter must record the exact base-model revision, tokenizer, target-module names, rank, scaling, and library version. Merging is useful for a fixed deployment, while keeping adapters separate is useful when many domains share one base model.

### **Post-Training and Preference Optimization** {#post-training-and-preference-optimization}

Pretraining gives a model broad language capability, and SFT demonstrates desired responses. Yet many prompts admit several grammatically valid answers with different levels of helpfulness, factual support, safety, style, or relevance. Preference optimization uses comparisons between candidate responses to shift probability toward responses judged better under an explicit collection protocol.

It is safer to call this **preference optimization** than to assume that a model becomes universally aligned. The learned behavior reflects who supplied the preferences, how prompts were sampled, what rubric was used, and which candidates annotators were allowed to compare.

#### **From Supervised Fine-Tuning to Preference Optimization** {#from-sft-to-preference-optimization}

SFT learns from one demonstrated target $y$ for prompt $x$. A preference example instead compares two candidates:

$$
(x,y_w,y_l)
$$

where $y_w$ is preferred (the winner or chosen response) and $y_l$ is less preferred (the loser or rejected response). The comparison provides relative information: it says which response is better under the rubric, not that either response is an ideal ground-truth answer.

```text
pretraining -> broad capability
SFT -> imitate demonstrations and response format
preference optimization -> rank plausible responses under a rubric
evaluation -> test whether the change generalizes beyond the preference set
```

Preference optimization normally begins from an SFT model. Starting from a policy that cannot already produce reasonable candidates makes both reward learning and policy optimization harder.

#### **Preference Data and Reward Modeling** {#preference-data-and-reward-modeling}

Preference data are often collected by showing annotators a prompt and two anonymized candidate responses. The protocol should randomize response order, permit ties or "both bad" labels when appropriate, define factuality and safety criteria, and measure agreement. If every comparison forces a winner, noise is introduced when the candidates are effectively equal.

A reward model $r_\phi(x,y)$ maps a prompt-response pair to a scalar. Under the Bradley-Terry model, the probability that $y_w$ is preferred to $y_l$ is:

$$
P(y_w\succ y_l\mid x)
=\sigma\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right)
$$

The pairwise reward-model loss is:

$$
\mathcal{L}_{\text{RM}}
=-\log \sigma\left(r_\phi(x,y_w)-r_\phi(x,y_l)\right)
$$

Only the reward difference matters. Adding the same constant to both scores changes nothing. A large positive margin means the model is confident that the chosen response should rank above the rejected one.

<details>
<summary>Python Pairwise Reward-Model Loss</summary>

```python
import torch
import torch.nn.functional as F

def pairwise_reward_loss(chosen_rewards, rejected_rewards):
    """
    chosen_rewards and rejected_rewards have shape [batch].
    The loss is small when the chosen response has the larger score.
    """
    reward_margin = chosen_rewards - rejected_rewards
    losses = -F.logsigmoid(reward_margin)

    metrics = {
        "loss": losses.mean(),
        "preference_accuracy": (reward_margin > 0).float().mean(),
        "mean_margin": reward_margin.mean(),
    }
    return metrics

chosen = torch.tensor([1.4, 0.2, 2.1])
rejected = torch.tensor([0.3, 0.5, 1.0])
print(pairwise_reward_loss(chosen, rejected))
```

</details>

Reward-model accuracy is not sufficient. The model may exploit length, formatting, politeness, citation-shaped text, or annotator-specific style rather than the intended quality. Evaluation should include held-out prompt domains, controlled length comparisons, adversarial candidates, calibration of margins, and direct human review.

#### **RLHF with PPO** {#rlhf-with-ppo}

Reinforcement Learning from Human Feedback (RLHF) commonly refers to a pipeline with three learned components: an SFT policy, a reward model trained from preferences, and a policy updated to obtain higher reward while remaining close to a reference model. Proximal Policy Optimization (PPO) is one algorithm used for the policy-update stage.

> ![InstructGPT supervised, reward-model, and PPO stages](assets/instructgpt-rlhf-pipeline.png)
>
> Source: [Training language models to follow instructions with human feedback](https://arxiv.org/abs/2203.02155)

The policy generates a response, the reward model scores it, and a KL penalty discourages the policy from moving too far from a fixed reference:

$$
R(x,y)
=r_\phi(x,y)
-\beta\,D_{\mathrm{KL}}
\left(\pi_\theta(\cdot\mid x)\,\|\,\pi_{\mathrm{ref}}(\cdot\mid x)\right)
$$

Here, $r_\phi$ is learned preference reward and $\beta$ controls conservatism. Without the KL term, the policy may exploit weaknesses in the reward model and produce responses unlike the distribution on which the reward model was trained.

PPO collects rollouts from an older policy $\pi_{\text{old}}$, estimates an advantage $A_t$, and constrains the probability-ratio update:

$$
r_t(\theta)
=\frac{\pi_\theta(a_t\mid s_t)}
{\pi_{\text{old}}(a_t\mid s_t)}
$$

$$
\mathcal{L}_{\text{PPO}}
=-\mathbb{E}_t\left[
\min\left(
r_t(\theta)A_t,
\operatorname{clip}(r_t(\theta),1-\epsilon,1+\epsilon)A_t
\right)
\right]
$$

If the new policy changes a token probability too much, clipping limits the immediate incentive from that sample. PPO does not literally guarantee a small global policy change; clipping, KL control, learning rate, rollout freshness, and value-function quality must work together.

<details>
<summary>Python PPO Clipped Policy Objective</summary>

```python
import torch

def ppo_policy_loss(
    new_log_probs,
    old_log_probs,
    advantages,
    clip_epsilon=0.2,
):
    # Probability ratio in a numerically stable log-probability form.
    ratio = torch.exp(new_log_probs - old_log_probs)

    unclipped = ratio * advantages
    clipped = torch.clamp(
        ratio,
        1.0 - clip_epsilon,
        1.0 + clip_epsilon,
    ) * advantages

    # PPO maximizes the conservative lower of the two surrogates.
    objective = torch.minimum(unclipped, clipped)
    return -objective.mean()
```

</details>

An NLP PPO system also needs a value model or value head, generalized advantage estimation, response masks, reward normalization, rollout generation, KL measurement, and distributed inference/training infrastructure. It is powerful because it can optimize samples generated by the current policy, but this online loop is expensive and sensitive to implementation details.

#### **Direct Preference Optimization** {#direct-preference-optimization}

Direct Preference Optimization (DPO) uses the same chosen/rejected data but avoids fitting a separate scalar reward model and avoids an explicit on-policy reinforcement-learning loop. It directly increases the policy's relative preference for $y_w$ over $y_l$, measured against a frozen reference policy.

> ![DPO compared with reward-model and PPO-based RLHF](assets/dpo-vs-rlhf.png)
>
> Source: [Direct Preference Optimization: Your Language Model is Secretly a Reward Model](https://arxiv.org/abs/2305.18290)

The DPO objective is:

$$
\mathcal{L}_{\text{DPO}}
=-\mathbb{E}_{(x,y_w,y_l)}
\log\sigma\left(
\beta
\left[
\log\frac{\pi_\theta(y_w\mid x)}{\pi_{\text{ref}}(y_w\mid x)}
-
\log\frac{\pi_\theta(y_l\mid x)}{\pi_{\text{ref}}(y_l\mid x)}
\right]
\right)
$$

Each sequence log-probability is the sum of response-token log-probabilities under teacher forcing. The bracket asks whether the trainable policy improves the chosen-vs-rejected log-probability margin more than the reference does. $\beta$ controls how strongly the solution is tied to the reference-policy geometry.

<details>
<summary>Python DPO Loss from Sequence Log-Probabilities</summary>

```python
import torch
import torch.nn.functional as F

def dpo_loss(
    policy_chosen_logp,
    policy_rejected_logp,
    reference_chosen_logp,
    reference_rejected_logp,
    beta=0.1,
):
    # Relative chosen-vs-rejected preference under each policy.
    policy_margin = policy_chosen_logp - policy_rejected_logp
    reference_margin = (
        reference_chosen_logp - reference_rejected_logp
    )

    logits = beta * (policy_margin - reference_margin)
    losses = -F.logsigmoid(logits)

    # Detached rewards are useful diagnostics, not a separate reward model.
    chosen_reward = beta * (
        policy_chosen_logp - reference_chosen_logp
    ).detach()
    rejected_reward = beta * (
        policy_rejected_logp - reference_rejected_logp
    ).detach()

    return {
        "loss": losses.mean(),
        "preference_accuracy": (
            chosen_reward > rejected_reward
        ).float().mean(),
        "reward_margin": (
            chosen_reward - rejected_reward
        ).mean(),
    }
```

</details>

DPO is operationally simpler than PPO, but not free of assumptions. It requires reliable sequence log-probabilities from both policy and reference, is sensitive to response-length normalization and data quality, and learns only from the fixed preference pairs it sees. It does not explore new responses online during optimization.

#### **DPO vs PPO vs Supervised Fine-Tuning** {#dpo-vs-ppo-vs-supervised-fine-tuning}

SFT, DPO, and PPO are not interchangeable optimizer names. They define different learning signals and data-generation loops.

| Method | Training signal | Extra learned model | Data loop | Main advantage | Main difficulty |
|---|---|---|---|---|---|
| SFT | imitate target responses | none | fixed demonstrations | stable and simple | one target does not express relative preference |
| DPO | chosen over rejected response | frozen reference policy | fixed offline pairs | simple preference training | limited by pair coverage and reference choice |
| PPO-based RLHF | learned reward on policy rollouts | reward model and usually value model | online generation | can optimize current-policy samples | expensive and instability-prone |

A common progression is SFT first, then DPO when good offline preference pairs exist, and PPO when online exploration and a sufficiently reliable reward signal justify the complexity. The decision should be empirical: compare task quality, preference win rate, factuality, safety slices, retained capability, KL movement, compute, and operational risk.

#### **Limitations of Preference Alignment** {#limitations-of-preference-alignment}

Preference labels are conditional judgments, not universal truth. Annotators may disagree across cultures, domains, expertise levels, or intended users. Pairwise data can favor verbosity because a longer response appears more complete, even when it contains more unsupported claims.

The training objective is also a proxy. A policy can improve reward-model score or preference accuracy while degrading properties not represented in the data. This is a form of Goodhart's law: once a measure becomes a target, pressure is applied to its blind spots.

| Failure mode | Why it appears | Mitigation |
|---|---|---|
| verbosity bias | longer answers look more helpful | length-controlled pairs and evaluation |
| sycophancy | agreement receives preference | disagreement and correction examples |
| reward hacking | policy exploits scorer shortcuts | adversarial audits and independent judges |
| preference overoptimization | training exceeds reliable data support | KL control, early stopping, held-out prompts |
| minority preference erasure | one aggregate label hides disagreement | subgroup analysis and plural rubrics |
| capability regression | narrow preference updates overwrite behavior | retained-capability suites and conservative updates |

Preference optimization must therefore be followed by broad evaluation rather than a single reward curve. Chapter [Evaluation in NLP](06-evaluation.html) explains task and human evaluation, while [Challenges, Risks, Ethics, and Safety](09-challenges.html) examines bias, misuse, privacy, and governance in greater depth.

### **Common Training Problems** {#common-training-problems}

Training failures surface through symptoms: a loss becomes `NaN`, throughput collapses at long sequences, validation quality stops improving, or a preference score rises while human-reviewed output gets worse. The same symptom can have several causes, so debugging should move from observable invariants toward more complex hypotheses.

```text
data and labels
-> tensor shapes, masks, and finite values
-> loss decomposition
-> gradients and optimizer updates
-> memory and distributed synchronization
-> validation slices and behavioral outputs
```

| Failure layer | Evidence to inspect first |
|---|---|
| data | decoded examples, labels, duplicates, split provenance |
| numerical | finite logits/loss/gradients, AMP scale, gradient norm |
| optimization | update norm, learning rate, warmup, tiny-batch overfit |
| systems | peak memory, worker logs, collective timing, skipped steps |
| generalization | learning curves, domain slices, retained capabilities |
| preference | length, style, factuality, reward margin, independent review |

#### **Vanishing and Exploding Gradients** {#vanishing-and-exploding-gradients}

Vanishing gradients happen when gradients become extremely small as they move backward through many layers or time steps. Earlier layers or earlier sequence positions receive almost no learning signal. Exploding gradients happen when gradients become extremely large, causing unstable updates, sudden loss spikes, or `NaN` values.

For a simplified chain of transformations, backpropagation repeatedly multiplies derivative terms:

$$
\frac{\partial L}{\partial h_1}
= \frac{\partial L}{\partial h_T}
\prod_{t=2}^{T} \frac{\partial h_t}{\partial h_{t-1}}
$$

If the multiplied terms are usually smaller than 1, the product shrinks toward zero. If they are usually larger than 1, the product grows rapidly. This is why very deep networks and recurrent models can be hard to train without architectural support.

```text
small derivative repeated many times -> vanishing gradient
large derivative repeated many times -> exploding gradient
```

In NLP, this problem is especially visible in RNN/LSTM/GRU training and in very deep Transformer training. Modern Transformers reduce the problem with residual connections, normalization layers, careful initialization, gradient clipping, and warmup schedules.

A practical symptom table:

| Problem | Training Symptom | Typical Fix |
|---|---|---|
| vanishing gradients | loss barely improves, early layers change little | residual connections, normalization, better initialization |
| exploding gradients | loss spikes, `NaN`, unstable updates | gradient clipping, lower learning rate, mixed precision care |
| unstable gradients | loss improves then suddenly collapses | clipping, warmup, smaller batch or learning rate |

<details>
<summary>Python Monitoring Gradient Norms</summary>

```python
import math
import torch
from torch.nn.utils import clip_grad_norm_

max_grad_norm = 1.0

for batch in dataloader:
    optimizer.zero_grad()

    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )
    loss = outputs.loss
    loss.backward()

    # Compute total gradient norm before clipping.
    total_norm_sq = 0.0
    for param in model.parameters():
        if param.grad is not None:
            param_norm = param.grad.detach().data.norm(2)
            total_norm_sq += param_norm.item() ** 2

    total_norm = math.sqrt(total_norm_sq)

    if total_norm > 10.0:
        print("Warning: very large gradient norm:", total_norm)

    # Clip gradients before the optimizer step.
    clip_grad_norm_(model.parameters(), max_norm=max_grad_norm)
    optimizer.step()
    scheduler.step()
```

</details>

The key is to monitor both loss and gradient norms. A loss curve tells us that training is unstable; gradient norms often tell us why.

#### **Learning Rate Too High or Too Low** {#learning-rate-too-high-or-too-low}

The learning rate controls how large the parameter update is. A good learning rate lets the model move steadily toward a better solution. A bad learning rate can make the model either unstable or painfully slow.

If the learning rate is too high, the optimizer may jump across useful regions of the loss surface. Training loss may oscillate, explode, or become `NaN`. If the learning rate is too low, training may look stable but barely improve.

| Learning Rate | Symptom | Interpretation |
|---|---|---|
| too high | loss jumps, diverges, or becomes `NaN` | update steps are too aggressive |
| too low | loss decreases extremely slowly | update steps are too cautious |
| reasonable | loss decreases with manageable noise | optimizer is making useful progress |
| good at first, bad later | early progress, later instability | schedule or warmup may need adjustment |

In Transformer fine-tuning, learning rates are usually small because the pretrained representation is already useful. A value like `2e-5` or `5e-5` may be reasonable for BERT-style fine-tuning, while training a small model from scratch may use a much larger learning rate such as `1e-3`.

A simple learning-rate range test tries increasing learning rates for a short run and records the loss. The useful range is often before the loss starts to explode.

<details>
<summary>Python Learning Rate Range Test Sketch</summary>

```python
import torch

start_lr = 1e-6
end_lr = 1e-2
num_steps = 100

lrs = torch.logspace(
    torch.log10(torch.tensor(start_lr)),
    torch.log10(torch.tensor(end_lr)),
    steps=num_steps
)

loss_history = []

for step, batch in enumerate(dataloader):
    if step >= num_steps:
        break

    # Update optimizer learning rate for this test step.
    for group in optimizer.param_groups:
        group["lr"] = float(lrs[step])

    optimizer.zero_grad()
    outputs = model(
        input_ids=batch["input_ids"],
        attention_mask=batch["attention_mask"],
        labels=batch["labels"]
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

    loss_history.append((float(lrs[step]), float(loss)))

    if not torch.isfinite(loss):
        print("Loss became unstable at lr=", float(lrs[step]))
        break

# In practice, plot lr vs loss and choose a value before sharp divergence.
print(loss_history[:5])
```

</details>

A learning rate problem is one of the first things to check because it can mimic many other failures. Before changing the model architecture, verify that the loss can decrease on a small batch with a reasonable learning rate.

#### **Numerical Instability and Out-of-Memory Errors** {#numerical-instability-and-out-of-memory-errors}

Numerical instability occurs when an intermediate value cannot be represented reliably or an invalid operation enters the computation graph. Common symptoms are `inf` logits, `NaN` loss, non-finite gradients, or a mixed-precision scaler that repeatedly skips optimizer steps.

Typical causes include an excessive learning rate, FP16 overflow or underflow, division by an empty loss mask, taking a logarithm of zero in a custom loss, an unstable softmax implementation, and corrupted input values. Framework cross-entropy functions should receive raw logits because they use numerically stable log-sum-exp calculations internally.

An out-of-memory (OOM) error is different: the requested tensors exceed available accelerator memory. Peak memory may occur during backward rather than forward, and it may vary with sequence length because attention and saved activations depend on token count. A run that fits ordinary batches may fail on one unusually long batch.

| Symptom | Likely cause | First response |
|---|---|---|
| loss is `NaN` immediately | invalid labels, mask, custom math | inspect one decoded batch and each loss term |
| loss becomes `NaN` later | high LR, exploding gradients, FP16 overflow | lower LR, clip, inspect AMP and gradient norms |
| OOM on first batch | model or micro-batch too large | reduce sequence/batch, use AMP or sharding |
| OOM after several steps | retained graph, logging tensor leak, fragmentation | detach stored tensors and profile allocations |
| OOM only on some batches | length outlier or dynamic shape | bucket/cap lengths and log token counts |

<details>
<summary>Python Finite-Value and Gradient Diagnostics</summary>

```python
import torch

def assert_finite_tensor(name, value):
    if value is not None and not torch.isfinite(value).all():
        raise FloatingPointError(f"{name} contains NaN or Inf")

def check_training_state(loss, model):
    # Check the scalar objective before backpropagation.
    assert_finite_tensor("loss", loss.detach())

    bad_parameters = []
    for name, parameter in model.named_parameters():
        if parameter.grad is None:
            continue
        if not torch.isfinite(parameter.grad).all():
            bad_parameters.append(name)

    if bad_parameters:
        preview = ", ".join(bad_parameters[:5])
        raise FloatingPointError(
            f"non-finite gradients in: {preview}"
        )

# Usage:
# loss.backward()
# check_training_state(loss, model)
# optimizer.step()
```

</details>

A disciplined OOM response identifies the dominant state. Reduce micro-batch or sequence length first, then use mixed precision for tensor bytes, activation checkpointing for saved activations, gradient accumulation to restore effective batch size, and FSDP/ZeRO when model states dominate. Repeatedly clearing the cache does not fix a genuinely oversized working set.

#### **Overfitting and Catastrophic Forgetting** {#overfitting-and-catastrophic-forgetting}

Overfitting in the training-problem section is the same phenomenon discussed in regularization, but here the focus is diagnosis. The model fits the training set better and better while validation performance stops improving.

The most common NLP causes are:

| Cause | Example |
|---|---|
| small labelled dataset | fine-tuning BERT on a few hundred examples |
| noisy labels | sentiment labels assigned inconsistently |
| shortcut features | model learns author/source style instead of task meaning |
| domain mismatch | validation/test data comes from a different domain |
| too many epochs | model keeps specializing after validation peak |

A useful debugging trick is to intentionally overfit a tiny subset. If the model cannot overfit 16 or 32 examples, there may be a bug in the labels, loss function, model output shape, or optimizer. If it overfits the tiny subset but fails validation, the training pipeline is probably working, and the issue is generalization.

<details>
<summary>Python Tiny-Batch Overfit Test</summary>

```python
# Take one small batch and train on it repeatedly.
# A healthy model should usually drive this tiny-batch loss down.
tiny_batch = next(iter(dataloader))

model.train()
for step in range(100):
    optimizer.zero_grad()

    outputs = model(
        input_ids=tiny_batch["input_ids"],
        attention_mask=tiny_batch["attention_mask"],
        labels=tiny_batch["labels"]
    )

    loss = outputs.loss
    loss.backward()
    optimizer.step()

    if step % 10 == 0:
        print(step, float(loss))
```

</details>

If tiny-batch loss does not decrease, check the data pipeline first. If tiny-batch loss decreases but validation remains poor, use regularization, more data, better splits, or a model better matched to the task.

Catastrophic forgetting happens when fine-tuning on a new task causes a pretrained model to lose useful behavior learned earlier. In NLP, this can appear when a general language model is fine-tuned too aggressively on a narrow dataset.

For example, a model fine-tuned on a small customer-support dataset may become better at support replies but worse at general summarization or broad question answering. The model did adapt, but the adaptation overwrote part of the general ability.

The tension is:

```text
plasticity: model should adapt to the new task
stability:  model should preserve useful old knowledge
```

Catastrophic forgetting is more likely when:

| Condition | Why It Matters |
|---|---|
| learning rate is too high | pretrained weights move too far |
| dataset is narrow | model over-specializes to one domain |
| training runs too long | repeated updates overwrite general patterns |
| all parameters are updated | no part of the model is protected |
| no general validation set is used | forgetting is not measured |

Common mitigations include small learning rates, early stopping, mixing some general data, regularization, replay data, PEFT methods such as LoRA/adapters, and freezing lower layers.

<details>
<summary>Python Monitoring Forgetting Across Two Evaluation Sets</summary>

```python
def evaluate_accuracy(model, dataloader):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for batch in dataloader:
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"]
            )
            predictions = outputs.logits.argmax(dim=-1)
            labels = batch["labels"]

            correct += (predictions == labels).sum().item()
            total += labels.numel()

    return correct / total

# Evaluate before fine-tuning.
general_before = evaluate_accuracy(model, general_eval_loader)
task_before = evaluate_accuracy(model, task_eval_loader)

# Fine-tune on the task-specific training set.
fine_tune(model, task_train_loader)

# Evaluate after fine-tuning.
general_after = evaluate_accuracy(model, general_eval_loader)
task_after = evaluate_accuracy(model, task_eval_loader)

print("general score change:", general_after - general_before)
print("task score change:", task_after - task_before)

if task_after > task_before and general_after < general_before:
    print("The model adapted to the task, but may have forgotten general behavior.")
```

</details>

The main lesson is that fine-tuning should be evaluated on what we want to gain and what we cannot afford to lose.

#### **Exposure Bias** {#exposure-bias}

Exposure bias appears in sequence generation when training conditions differ from inference conditions. During teacher forcing, the decoder receives the correct previous token. During inference, the decoder receives its own previous output.

```text
training step t:
correct previous token -> predict next token

inference step t:
model previous token -> predict next token
```

If the model makes a small mistake early during inference, the next step receives a context that may never have appeared during teacher-forced training. This can compound into repetitive, off-topic, or malformed output.

A simple example:

```text
target:       The cat sat on the mat.
model output: The cat sat sat sat ...
```

The token-level training loss may look acceptable because the model was trained under clean histories. But generation quality depends on how the model behaves under its own histories.

Common mitigations include scheduled sampling, better decoding strategies, sequence-level objectives, reinforcement-style fine-tuning, data augmentation, and instruction tuning. For many modern LLM workflows, exposure bias is handled partly by large-scale pretraining, high-quality instruction tuning, and decoding controls such as repetition penalty or nucleus sampling.

<details>
<summary>Python Demonstrating Teacher-Forced vs Autoregressive Inputs</summary>

```python
BOS = "<bos>"
EOS = "<eos>"

target = [BOS, "The", "cat", "sat", "on", "the", "mat", EOS]

# Teacher forcing: each decoder input is the correct previous token.
teacher_forced_inputs = target[:-1]
teacher_forced_labels = target[1:]

print("training inputs: ", teacher_forced_inputs)
print("training labels: ", teacher_forced_labels)

# Autoregressive inference: each next input depends on the model's own output.
generated = [BOS]
for step in range(5):
    previous_context = generated

    # This is a placeholder for model.generate_next_token(previous_context).
    next_token = "sat" if step >= 2 else target[step + 1]
    generated.append(next_token)

print("inference context grows from model outputs:", generated)
```

</details>

Exposure bias is not solved by monitoring training loss alone. Always inspect generated samples and evaluate sequence-level behavior.

| Problem | Main Symptom | First Checks | Common Fixes |
|---|---|---|---|
| vanishing gradients | slow or no learning in deep/sequence models | gradient norms, layer updates | residuals, normalization, initialization |
| exploding gradients | loss spikes or `NaN` | gradient norms, learning rate | clipping, lower LR, warmup |
| bad learning rate | divergence or very slow learning | LR range, small-batch test | tune LR, scheduler, warmup |
| overfitting | train improves, validation worsens | split quality, tiny-batch test | dropout, weight decay, early stopping |
| catastrophic forgetting | new task improves, old ability drops | evaluate old and new tasks | PEFT, replay, lower LR, freeze layers |
| exposure bias | generated text degrades after early mistakes | inspect generations | decoding control, scheduled sampling, SFT/RLHF |

#### **Reward Hacking and Preference Overoptimization** {#reward-hacking-and-preference-overoptimization}

Reward hacking occurs when a model finds behavior that scores well under a learned or programmed reward without satisfying the intended objective. In language systems, this can mean producing verbose answers, imitating citation format without evidence, using phrases correlated with high ratings, or avoiding difficult content in a way the scorer mistakes for safety.

Preference overoptimization occurs when additional optimization keeps improving the training proxy but moves beyond the region where that proxy reliably predicts human judgment. The reward model was trained on a limited candidate distribution; as the policy changes, it may generate outputs outside that distribution and exploit extrapolation errors.

```text
early training: reward score rises and human quality rises
later training: reward score still rises but human quality plateaus or falls
```

Useful controls include a reference-policy KL penalty, conservative learning rates, early stopping on held-out preference prompts, independent human or model judges, length-controlled analysis, and retained-capability tests. For high-stakes behavior, evaluate factual support and policy compliance directly rather than using reward alone.

| Signal | Healthy interpretation | Warning sign |
|---|---|---|
| training reward | proxy improves | rises while independent quality falls |
| KL from reference | controlled adaptation | sudden or unbounded movement |
| response length | task-appropriate | grows merely to obtain reward |
| preference margin | separates known pairs | confidence rises on noisy or ambiguous pairs |
| held-out win rate | generalizes to new prompts | only training-pair accuracy improves |

The deeper lesson is that every training target is incomplete. Preference optimization requires the same scientific discipline as ordinary supervised learning: protected evaluation data, ablations, uncertainty, subgroup analysis, and inspection of concrete failures.

### **Practical End-to-End Training Workflow** {#practical-end-to-end-training-workflow}

A reliable workflow treats model quality, systems feasibility, and post-training behavior as linked experiments. Scaling a broken objective only produces an expensive failure, while changing several training stages at once makes the cause of improvement impossible to identify.

Start with the smallest configuration that can falsify the pipeline: decode inputs and labels, overfit a tiny batch, compare with a simple baseline, and record a deterministic checkpoint. Only then add mixed precision, accumulation, distributed execution, PEFT, or preference optimization one change at a time.

A practical NLP training workflow should make mistakes visible early. The goal is not only to train a model, but to build a loop where data bugs, optimization problems, and generalization problems are easy to diagnose.

A reliable workflow is:

```text
1. Define the task and output format
2. Build a tiny baseline
3. Verify tokenization, labels, padding, and masks
4. Run a tiny-batch overfit test
5. Choose objective, metric, optimizer, and scheduler
6. Train with validation monitoring
7. Inspect errors and generated outputs
8. Adjust regularization or data strategy
9. Save the best checkpoint
10. Evaluate on held-out or domain-shifted data
```

The first step is to define the modeling target precisely. A classification model needs label definitions. A token classifier needs alignment between tokens and labels. A seq2seq model needs clear source-target formatting. A language model needs a decision about which tokens contribute to the loss.

Before training a large model, build a small baseline. This could be logistic regression with TF-IDF, a frozen pretrained encoder with a linear head, or a tiny neural model. The baseline gives a sanity check: if a huge model cannot beat a simple baseline, the issue may be data quality, metric choice, or task formulation.

The next step is data verification. Many NLP training failures come from quiet data bugs:

| Check | What Can Go Wrong |
|---|---|
| tokenization | truncation removes important text |
| labels | label IDs do not match label names |
| padding | model attends to `<pad>` tokens |
| loss masking | model is trained on prompt or pad tokens unintentionally |
| split | train and validation distributions differ too much or leak data |
| metrics | accuracy hides class imbalance |

A minimal training skeleton looks like this:

<details>
<summary>Python Practical Training Loop Skeleton</summary>

```python
import torch
from torch.nn.utils import clip_grad_norm_

max_grad_norm = 1.0
best_valid_metric = -float("inf")
best_state_dict = None
patience = 3
bad_epochs = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss.backward()

        # Stabilize training before the parameter update.
        clip_grad_norm_(model.parameters(), max_grad_norm)

        optimizer.step()
        scheduler.step()

        train_loss += float(loss)

    # Validation phase: dropout off, no gradient computation.
    model.eval()
    valid_metric = evaluate(model, valid_loader)

    print(
        f"epoch={epoch + 1}",
        f"train_loss={train_loss / len(train_loader):.4f}",
        f"valid_metric={valid_metric:.4f}"
    )

    # Save the best checkpoint according to validation performance.
    if valid_metric > best_valid_metric:
        best_valid_metric = valid_metric
        best_state_dict = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        bad_epochs = 0
    else:
        bad_epochs += 1

    if bad_epochs >= patience:
        print("Early stopping.")
        break

# Restore the best model before final evaluation.
model.load_state_dict(best_state_dict)
test_metric = evaluate(model, test_loader)
print("test metric:", test_metric)
```

</details>

For generation tasks, workflow must include qualitative inspection. A model can have acceptable token-level loss but still generate repetitive, unsafe, too short, too long, or off-task text. Keep a small fixed set of prompts and inspect outputs after each major training change.

<details>
<summary>Python Fixed Prompt Generation Check</summary>

```python
fixed_prompts = [
    "Summarize: Neural NLP models require careful optimization.",
    "Translate to French: I enjoy studying natural language processing.",
    "Classify sentiment: The explanation was clear and useful."
]

model.eval()
for prompt in fixed_prompts:
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **inputs,
        max_new_tokens=64,
        do_sample=False
    )

    print("PROMPT:", prompt)
    print("OUTPUT:", tokenizer.decode(generated_ids[0], skip_special_tokens=True))
    print("---")
```

</details>

A practical debugging checklist:

| Symptom | Check First | Likely Action |
|---|---|---|
| training loss does not decrease | labels, loss shape, optimizer step | run tiny-batch overfit test |
| loss becomes `NaN` | learning rate, mixed precision, gradient norm | lower LR, clip gradients, inspect data |
| validation poor but training good | overfitting, split mismatch | regularization, early stopping, more data |
| predicts majority class | class distribution, metric choice | weighted loss, resampling, F1/recall |
| generated text repetitive | decoding, exposure bias, data duplication | repetition penalty, sampling, better data |
| fine-tuning unstable | LR, warmup, batch size | smaller LR, warmup, AdamW, PEFT |
| old capability drops | catastrophic forgetting | evaluate general set, freeze/PEFT/replay |

The overall idea is simple: do not treat training as a single command. Treat it as an experiment loop. Define the target, verify the data, prove the model can learn a tiny case, train with monitoring, inspect failure modes, and only then scale up.

Training and optimization connect model design to real performance. The architecture defines what the model can express; training determines whether it actually learns useful behavior from data.

For a modern generative model, the full lifecycle can be organized into explicit gates:

| Gate | Required evidence before proceeding |
|---|---|
| data gate | provenance, deduplication, split integrity, representative slices |
| objective gate | labels and masks correspond to intended behavior |
| optimization gate | tiny batch overfits; loss and gradients remain finite |
| systems gate | global batch, update count, memory, and distributed metrics verified |
| adaptation gate | SFT/full/PEFT comparison on held-out task data |
| preference gate | pair quality, agreement, reward/DPO diagnostics, KL movement |
| release gate | task, robustness, safety, retained-capability, and latency evaluation |

Experiment records should include more than hyperparameters. Save the exact dataset revision, tokenizer files, prompt template, model commit, adapter configuration, quantization configuration, random seeds, hardware topology, library versions, global batch derivation, scheduler step count, and evaluation code.

<details>
<summary>Python Reproducible Training Run Metadata</summary>

```python
from dataclasses import asdict, dataclass
import json
import platform
import torch

@dataclass
class RunMetadata:
    base_model: str
    dataset_revision: str
    tokenizer_revision: str
    objective: str
    adaptation_method: str
    micro_batch_size: int
    accumulation_steps: int
    data_parallel_size: int
    learning_rate: float
    seed: int

    @property
    def global_batch_size(self):
        return (
            self.micro_batch_size
            * self.accumulation_steps
            * self.data_parallel_size
        )

metadata = RunMetadata(
    base_model=model_name,
    dataset_revision=dataset_revision,
    tokenizer_revision=tokenizer_revision,
    objective="response_only_sft",
    adaptation_method="lora",
    micro_batch_size=2,
    accumulation_steps=8,
    data_parallel_size=4,
    learning_rate=2e-4,
    seed=42,
)

record = asdict(metadata)
record["global_batch_size"] = metadata.global_batch_size
record["torch_version"] = torch.__version__
record["python_version"] = platform.python_version()

with open("run_metadata.json", "w", encoding="utf-8") as file:
    json.dump(record, file, indent=2)
```

</details>

Final evaluation should match the stage that was changed. A systems optimization must preserve the numerical baseline; PEFT must be compared with frozen and full-fine-tuning baselines; preference optimization must be checked against SFT on both preferred behavior and retained capability. Chapter 06 provides the evaluation framework needed for those comparisons.

The chapter's main distinction is now clear: objectives define the learning signal, optimizers transform gradients, scaling methods make the computation feasible, adaptation strategies determine which knowledge and parameters change, and preference methods alter relative response selection. Treating these as separate decisions makes training easier to reason about, reproduce, and debug.
